In [1]:
# Cell 1: TALENTMATCH project setup

from pathlib import Path

def _resolve_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "TALENTMATCH_AUDIT",
        Path.cwd() / "talentmatch",
        Path("/content/TALENTMATCH_AUDIT"),
        Path("/content/talentmatch"),
    ]
    for candidate in candidates:
        if (candidate / "data" / "synthetic").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate TALENTMATCH project directory. "
        "Run this notebook from the project folder."
    )


# Project directories
PROJECT_DIR = _resolve_project_dir()
SYNTHETIC_DIR = PROJECT_DIR / "data" / "synthetic"
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

for directory in [SYNTHETIC_DIR, EVALUATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Define relevance labels
ANNOTATION_RUBRIC = """
TALENTMATCH — Relevance Annotation Rubric

Label 0 — Low relevance
Little demonstrated alignment between resume evidence
and the job requirements.

Label 1 — Partial relevance
Some relevant skills or experience, but important
requirements have limited or no supporting evidence.

Label 2 — Strong relevance
Strong alignment between demonstrated resume evidence
and the job requirements.

Rules:
1. Evaluate only the supplied resume and job description.
2. Do not infer skills without supporting evidence.
3. Missing evidence does not prove missing ability.
4. Do not consider names, gender, religion, or other
   protected characteristics.
5. Record a short explanation for every label.
6. Review ambiguous pairs manually.
"""

RUBRIC_PATH = PROJECT_DIR / "annotation_rubric.md"
if not RUBRIC_PATH.exists():
    RUBRIC_PATH.write_text(ANNOTATION_RUBRIC.strip(), encoding="utf-8")
else:
    print("Existing annotation rubric preserved.")

print("Project:", PROJECT_DIR)
print("Synthetic data:", SYNTHETIC_DIR)
print("Evaluation data:", EVALUATION_DIR)
print("Annotation rubric:", RUBRIC_PATH)
print("\nDirectory setup successful.")
print("\nRelevance labels: 0 = Low | 1 = Partial | 2 = Strong")

Project: /content/talentmatch
Synthetic data: /content/talentmatch/data/synthetic
Evaluation data: /content/talentmatch/data/evaluation
Annotation rubric: /content/talentmatch/annotation_rubric.md

Directory setup successful.

Relevance labels: 0 = Low | 1 = Partial | 2 = Strong


In [2]:
# Cell 2: Create synthetic resumes and job descriptions

import pandas as pd

# 10 synthetic resumes
resumes = [
    {
        "resume_id": "R001",
        "profile": "Python Developer",
        "resume_text": """
        Python developer with experience in Python, Flask, REST APIs,
        SQL, Git, and unit testing. Built web applications and automated
        data-processing workflows. Familiar with PostgreSQL and Docker.
        """
    },
    {
        "resume_id": "R002",
        "profile": "Data Analyst",
        "resume_text": """
        Data analyst skilled in SQL, Python, pandas, NumPy, Excel,
        and Power BI. Experienced in exploratory data analysis,
        dashboard creation, data cleaning, and reporting.
        """
    },
    {
        "resume_id": "R003",
        "profile": "Machine Learning Engineer",
        "resume_text": """
        Machine learning practitioner with Python, scikit-learn,
        pandas, NumPy, model evaluation, feature engineering,
        classification, regression, and cross-validation experience.
        """
    },
    {
        "resume_id": "R004",
        "profile": "NLP Engineer",
        "resume_text": """
        NLP practitioner experienced with Transformers, tokenization,
        text classification, sentence embeddings, semantic search,
        Hugging Face, and PyTorch. Built text-processing pipelines.
        """
    },
    {
        "resume_id": "R005",
        "profile": "Backend Developer",
        "resume_text": """
        Backend developer experienced in Python, FastAPI, REST APIs,
        PostgreSQL, SQL, Git, Docker, and automated testing.
        Built API endpoints and database-backed services.
        """
    },
    {
        "resume_id": "R006",
        "profile": "Frontend Developer",
        "resume_text": """
        Frontend developer skilled in HTML, CSS, JavaScript,
        React, responsive design, accessibility, and Git.
        Built interactive web interfaces and reusable UI components.
        """
    },
    {
        "resume_id": "R007",
        "profile": "Computer Vision Engineer",
        "resume_text": """
        Computer vision practitioner using Python, PyTorch,
        CNNs, image augmentation, transfer learning, and OpenCV.
        Built image classification experiments and evaluated
        models using precision, recall, and confusion matrices.
        """
    },
    {
        "resume_id": "R008",
        "profile": "Cloud Developer",
        "resume_text": """
        Cloud-oriented developer familiar with Linux, Git,
        Docker, CI/CD, AWS, monitoring, and deployment automation.
        Built and maintained containerized application workflows.
        """
    },
    {
        "resume_id": "R009",
        "profile": "Business Analyst",
        "resume_text": """
        Business analyst skilled in requirements gathering,
        stakeholder communication, Excel, SQL, documentation,
        process analysis, and reporting. Created business dashboards.
        """
    },
    {
        "resume_id": "R010",
        "profile": "Junior Software Developer",
        "resume_text": """
        Junior developer with foundational knowledge of Python,
        Java, SQL, Git, object-oriented programming, data structures,
        debugging, and software testing. Built small academic projects.
        """
    }
]

# 10 synthetic job descriptions
jobs = [
    {
        "job_id": "J001",
        "job_title": "Python Backend Developer",
        "job_description": """
        Required: Python, REST APIs, SQL, Git, and automated testing.
        Preferred: FastAPI, PostgreSQL, Docker, and application deployment.
        Responsibilities include building and maintaining backend services.
        """
    },
    {
        "job_id": "J002",
        "job_title": "Data Analyst",
        "job_description": """
        Required: SQL, Python, data cleaning, and exploratory data analysis.
        Preferred: pandas, Excel, Power BI, dashboard creation, and reporting.
        Responsibilities include analyzing data and communicating insights.
        """
    },
    {
        "job_id": "J003",
        "job_title": "Machine Learning Engineer",
        "job_description": """
        Required: Python, scikit-learn, feature engineering,
        model evaluation, and machine learning fundamentals.
        Preferred: pandas, NumPy, cross-validation, classification,
        and regression.
        """
    },
    {
        "job_id": "J004",
        "job_title": "NLP Engineer",
        "job_description": """
        Required: Python, NLP, tokenization, Transformers,
        and text classification.
        Preferred: Hugging Face, PyTorch, sentence embeddings,
        and semantic search.
        """
    },
    {
        "job_id": "J005",
        "job_title": "Computer Vision Engineer",
        "job_description": """
        Required: Python, CNNs, image classification, and PyTorch.
        Preferred: OpenCV, image augmentation, transfer learning,
        and model evaluation.
        """
    },
    {
        "job_id": "J006",
        "job_title": "Frontend Developer",
        "job_description": """
        Required: HTML, CSS, JavaScript, and responsive web design.
        Preferred: React, accessibility, Git, and reusable UI components.
        Responsibilities include building accessible web interfaces.
        """
    },
    {
        "job_id": "J007",
        "job_title": "Cloud and DevOps Engineer",
        "job_description": """
        Required: Linux, Docker, CI/CD, and deployment automation.
        Preferred: AWS, Git, monitoring, and container orchestration.
        Responsibilities include improving deployment reliability.
        """
    },
    {
        "job_id": "J008",
        "job_title": "Business Data Analyst",
        "job_description": """
        Required: SQL, Excel, reporting, and data analysis.
        Preferred: Power BI, stakeholder communication,
        requirements gathering, and dashboard development.
        """
    },
    {
        "job_id": "J009",
        "job_title": "Junior Software Engineer",
        "job_description": """
        Required: programming fundamentals, data structures,
        debugging, Git, and software testing.
        Preferred: Python, Java, SQL, and object-oriented programming.
        """
    },
    {
        "job_id": "J010",
        "job_title": "Applied AI Engineer",
        "job_description": """
        Required: Python, machine learning, model evaluation,
        and experience implementing AI applications.
        Preferred: PyTorch, Transformers, NLP, sentence embeddings,
        and semantic search.
        """
    }
]

# Convert to DataFrames
resumes_df = pd.DataFrame(resumes)
jobs_df = pd.DataFrame(jobs)

# Validate structure and uniqueness
assert len(resumes_df) == 10
assert len(jobs_df) == 10
assert resumes_df["resume_id"].is_unique
assert jobs_df["job_id"].is_unique
assert resumes_df["resume_text"].str.strip().ne("").all()
assert jobs_df["job_description"].str.strip().ne("").all()

# Save CSV files
resumes_path = SYNTHETIC_DIR / "resumes.csv"
jobs_path = SYNTHETIC_DIR / "job_descriptions.csv"

resumes_df.to_csv(resumes_path, index=False)
jobs_df.to_csv(jobs_path, index=False)

print("Resumes created:", len(resumes_df))
print("Job descriptions created:", len(jobs_df))
print("Resume file:", resumes_path)
print("Job file:", jobs_path)

print("\nResume preview:")
display(resumes_df[["resume_id", "profile"]].head())

print("\nJob description preview:")
display(jobs_df[["job_id", "job_title"]].head())

print("\nAll validation checks passed.")

Resumes created: 10
Job descriptions created: 10
Resume file: /content/talentmatch/data/synthetic/resumes.csv
Job file: /content/talentmatch/data/synthetic/job_descriptions.csv

Resume preview:


,resume_id,profile
0,R001,Python Developer
1,R002,Data Analyst
2,R003,Machine Learning Engineer
3,R004,NLP Engineer
4,R005,Backend Developer



Job description preview:


,job_id,job_title
0,J001,Python Backend Developer
1,J002,Data Analyst
2,J003,Machine Learning Engineer
3,J004,NLP Engineer
4,J005,Computer Vision Engineer



All validation checks passed.


In [3]:
# Cell 3: Create manually reviewed resume-job evaluation pairs

import pandas as pd

evaluation_pairs = [
    # Strong matches
    ("R001", "J001", 2, "Python, REST APIs, SQL, Git, and testing align with requirements."),
    ("R002", "J002", 2, "SQL, Python, data cleaning, analysis, and reporting align."),
    ("R003", "J003", 2, "Python, scikit-learn, feature engineering, and model evaluation align."),
    ("R004", "J004", 2, "Python, NLP, Transformers, tokenization, and text classification align."),
    ("R007", "J005", 2, "Python, CNNs, PyTorch, and image classification align."),
    ("R006", "J006", 2, "HTML, CSS, JavaScript, React, and responsive design align."),
    ("R008", "J007", 2, "Linux, Docker, CI/CD, and deployment automation align."),
    ("R009", "J008", 2, "SQL, Excel, reporting, analysis, and stakeholder communication align."),
    ("R010", "J009", 2, "Programming fundamentals, data structures, debugging, and testing align."),

    # Partial matches
    ("R005", "J001", 2, "Python, REST APIs, SQL, Git, testing, FastAPI, PostgreSQL, and Docker align."),
    ("R003", "J010", 2, "Python, machine learning, model evaluation, PyTorch, and NLP-related skills provide substantial alignment."),
    ("R004", "J010", 2, "Python, Transformers, NLP, sentence embeddings, and semantic search align."),
    ("R002", "J008", 2, "SQL, Excel, reporting, and data analysis align; preferred skills include some matches."),
    ("R001", "J009", 1, "Python, Git, and testing align, but data structures and Java evidence are limited."),
    ("R010", "J001", 1, "Python, SQL, and Git align, but backend API and deployment evidence is limited."),
    ("R003", "J002", 1, "Python and data analysis tools overlap, but reporting and dashboard evidence is limited."),
    ("R005", "J007", 1, "Docker and deployment-related experience overlap, but Linux and CI/CD evidence is limited."),
    ("R006", "J004", 0, "Frontend skills provide little direct evidence for the NLP requirements."),
    ("R009", "J005", 0, "Business analysis skills provide little direct evidence for computer vision requirements."),
    ("R002", "J007", 0, "Data analysis skills provide little direct evidence for cloud and DevOps requirements.")
]

evaluation_df = pd.DataFrame(
    evaluation_pairs,
    columns=[
        "resume_id",
        "job_id",
        "relevance_label",
        "label_reason"
    ]
)

# Validate IDs and labels
assert evaluation_df["resume_id"].isin(resumes_df["resume_id"]).all()
assert evaluation_df["job_id"].isin(jobs_df["job_id"]).all()
assert evaluation_df["relevance_label"].isin([0, 1, 2]).all()
assert evaluation_df["label_reason"].str.strip().ne("").all()

# Ensure each resume-job pair is unique
assert not evaluation_df.duplicated(["resume_id", "job_id"]).any()

# Save reviewed evaluation pairs
evaluation_path = EVALUATION_DIR / "reviewed_pairs.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print("Evaluation pairs:", len(evaluation_df))
print("Label distribution:")
print(evaluation_df["relevance_label"].value_counts().sort_index())

print("\nSaved to:", evaluation_path)

display(evaluation_df.head(10))

print("\nValidation checks passed.")

Evaluation pairs: 20
Label distribution:
relevance_label
0     3
1     4
2    13
Name: count, dtype: int64

Saved to: /content/talentmatch/data/evaluation/reviewed_pairs.csv


,resume_id,job_id,relevance_label,label_reason
0,R001,J001,2,"Python, REST APIs, SQL, Git, and testing align..."
1,R002,J002,2,"SQL, Python, data cleaning, analysis, and repo..."
2,R003,J003,2,"Python, scikit-learn, feature engineering, and..."
3,R004,J004,2,"Python, NLP, Transformers, tokenization, and t..."
4,R007,J005,2,"Python, CNNs, PyTorch, and image classificatio..."
5,R006,J006,2,"HTML, CSS, JavaScript, React, and responsive d..."
6,R008,J007,2,"Linux, Docker, CI/CD, and deployment automatio..."
7,R009,J008,2,"SQL, Excel, reporting, analysis, and stakehold..."
8,R010,J009,2,"Programming fundamentals, data structures, deb..."
9,R005,J001,2,"Python, REST APIs, SQL, Git, testing, FastAPI,..."



Validation checks passed.


In [4]:
# Cell 4: Audit resume-job evaluation labels

audit_df = (
    evaluation_df
    .merge(resumes_df, on="resume_id", how="left")
    .merge(jobs_df, on="job_id", how="left")
)

# Check missing joins
assert audit_df["resume_text"].notna().all()
assert audit_df["job_description"].notna().all()

# Display each pair for manual review
for index, row in audit_df.iterrows():
    print("=" * 80)
    print(f"Pair: {row['resume_id']} + {row['job_id']}")
    print(f"Resume profile: {row['profile']}")
    print(f"Job title: {row['job_title']}")
    print(f"Current label: {row['relevance_label']}")
    print(f"Reason: {row['label_reason']}")

    print("\nRESUME:")
    print(row["resume_text"].strip())

    print("\nJOB DESCRIPTION:")
    print(row["job_description"].strip())

    print("\nReview: Does the label match the supplied evidence?")
    print("0 = Low | 1 = Partial | 2 = Strong")

print("\nTotal pairs available for review:", len(audit_df))

Pair: R001 + J001
Resume profile: Python Developer
Job title: Python Backend Developer
Current label: 2
Reason: Python, REST APIs, SQL, Git, and testing align with requirements.

RESUME:
Python developer with experience in Python, Flask, REST APIs,
        SQL, Git, and unit testing. Built web applications and automated
        data-processing workflows. Familiar with PostgreSQL and Docker.

JOB DESCRIPTION:
Required: Python, REST APIs, SQL, Git, and automated testing.
        Preferred: FastAPI, PostgreSQL, Docker, and application deployment.
        Responsibilities include building and maintaining backend services.

Review: Does the label match the supplied evidence?
0 = Low | 1 = Partial | 2 = Strong
Pair: R002 + J002
Resume profile: Data Analyst
Job title: Data Analyst
Current label: 2
Reason: SQL, Python, data cleaning, analysis, and reporting align.

RESUME:
Data analyst skilled in SQL, Python, pandas, NumPy, Excel,
        and Power BI. Experienced in exploratory data analysis,

In [5]:
# Cell 5: Apply review corrections and validate evaluation data

# Correct R003 + J010
mask = (
    (evaluation_df["resume_id"] == "R003") &
    (evaluation_df["job_id"] == "J010")
)

evaluation_df.loc[mask, "relevance_label"] = 1
evaluation_df.loc[mask, "label_reason"] = (
    "Python and machine learning fundamentals align, "
    "but explicit AI application implementation evidence "
    "and several preferred skills are missing."
)

# Correct R004 + J010
mask = (
    (evaluation_df["resume_id"] == "R004") &
    (evaluation_df["job_id"] == "J010")
)

evaluation_df.loc[mask, "relevance_label"] = 1
evaluation_df.loc[mask, "label_reason"] = (
    "Strong NLP and Transformer overlap, but Python, "
    "general machine learning, and AI application "
    "implementation are not explicitly evidenced."
)

# Correct the explanation for R004 + J004
mask = (
    (evaluation_df["resume_id"] == "R004") &
    (evaluation_df["job_id"] == "J004")
)

evaluation_df.loc[mask, "label_reason"] = (
    "Transformers, tokenization, text classification, "
    "and preferred NLP tools align. Python is not "
    "explicitly stated in the resume."
)

# Validate dataset
assert len(evaluation_df) == 20
assert evaluation_df["resume_id"].isin(resumes_df["resume_id"]).all()
assert evaluation_df["job_id"].isin(jobs_df["job_id"]).all()
assert evaluation_df["relevance_label"].isin([0, 1, 2]).all()
assert evaluation_df["label_reason"].str.strip().ne("").all()
assert not evaluation_df.duplicated(["resume_id", "job_id"]).any()

# Save updated annotations
evaluation_path = EVALUATION_DIR / "reviewed_pairs.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print("Evaluation pairs:", len(evaluation_df))
print("\nUpdated label distribution:")
print(evaluation_df["relevance_label"].value_counts().sort_index())

print("\nCorrected pairs:")
display(
    evaluation_df[
        (
            (evaluation_df["resume_id"] == "R003") &
            (evaluation_df["job_id"] == "J010")
        ) |
        (
            (evaluation_df["resume_id"] == "R004") &
            (evaluation_df["job_id"] == "J010")
        ) |
        (
            (evaluation_df["resume_id"] == "R004") &
            (evaluation_df["job_id"] == "J004")
        )
    ]
)

print("\nSaved:", evaluation_path)
print("All validation checks passed.")

Evaluation pairs: 20

Updated label distribution:
relevance_label
0     3
1     6
2    11
Name: count, dtype: int64

Corrected pairs:


,resume_id,job_id,relevance_label,label_reason
3,R004,J004,2,"Transformers, tokenization, text classificatio..."
10,R003,J010,1,Python and machine learning fundamentals align...
11,R004,J010,1,"Strong NLP and Transformer overlap, but Python..."



Saved: /content/talentmatch/data/evaluation/reviewed_pairs.csv
All validation checks passed.


In [6]:
# Cell 6: Verify saved dataset artifacts

import pandas as pd

# Reload files from disk to verify saved artifacts
resumes_check = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_check = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
evaluation_check = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Required columns
assert {"resume_id", "resume_text"}.issubset(resumes_check.columns)
assert {"job_id", "job_title", "job_description"}.issubset(jobs_check.columns)
assert {
    "resume_id",
    "job_id",
    "relevance_label",
    "label_reason"
}.issubset(evaluation_check.columns)

# Check records and unique IDs
assert len(resumes_check) == 10
assert len(jobs_check) == 10
assert len(evaluation_check) == 20

assert resumes_check["resume_id"].is_unique
assert jobs_check["job_id"].is_unique
assert not evaluation_check.duplicated(["resume_id", "job_id"]).any()

# Check text and labels
assert resumes_check["resume_text"].notna().all()
assert resumes_check["resume_text"].str.strip().ne("").all()

assert jobs_check["job_description"].notna().all()
assert jobs_check["job_description"].str.strip().ne("").all()

assert evaluation_check["resume_id"].isin(resumes_check["resume_id"]).all()
assert evaluation_check["job_id"].isin(jobs_check["job_id"]).all()
assert evaluation_check["relevance_label"].isin([0, 1, 2]).all()
assert evaluation_check["label_reason"].notna().all()
assert evaluation_check["label_reason"].str.strip().ne("").all()

# Report audit results
print("RESUMES:", len(resumes_check))
print("JOB DESCRIPTIONS:", len(jobs_check))
print("REVIEWED PAIRS:", len(evaluation_check))

print("\nLABEL DISTRIBUTION:")
print(evaluation_check["relevance_label"].value_counts().sort_index())

print("\nMISSING VALUES:")
print({
    "resumes": int(resumes_check.isna().sum().sum()),
    "jobs": int(jobs_check.isna().sum().sum()),
    "evaluation": int(evaluation_check.isna().sum().sum())
})

print("\nVERIFIED FILES:")
print(SYNTHETIC_DIR / "resumes.csv")
print(SYNTHETIC_DIR / "job_descriptions.csv")
print(EVALUATION_DIR / "reviewed_pairs.csv")
print(PROJECT_DIR / "annotation_rubric.md")

print("\nMilestone 1 audit passed.")

RESUMES: 10
JOB DESCRIPTIONS: 10
REVIEWED PAIRS: 20

LABEL DISTRIBUTION:
relevance_label
0     3
1     6
2    11
Name: count, dtype: int64

MISSING VALUES:
{'resumes': 0, 'jobs': 0, 'evaluation': 0}

VERIFIED FILES:
/content/talentmatch/data/synthetic/resumes.csv
/content/talentmatch/data/synthetic/job_descriptions.csv
/content/talentmatch/data/evaluation/reviewed_pairs.csv
/content/talentmatch/annotation_rubric.md

Milestone 1 audit passed.


In [7]:
# Cell 7: Prepare text-matching baseline

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load verified project data
resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
evaluation_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Convert resume and job text into TF-IDF vectors
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

resume_vectors = tfidf.fit_transform(resumes_df["resume_text"])
job_vectors = tfidf.transform(jobs_df["job_description"])

# Compute similarity for every resume-job combination
similarity_matrix = cosine_similarity(
    resume_vectors,
    job_vectors
)

# Store scores with IDs for easy lookup
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=resumes_df["resume_id"],
    columns=jobs_df["job_id"]
)

print("Resume vectors:", resume_vectors.shape)
print("Job vectors:", job_vectors.shape)
print("Similarity matrix:", similarity_df.shape)

print("\nExample: R001 matched against all jobs")
print(
    similarity_df.loc["R001"]
    .sort_values(ascending=False)
    .round(3)
)

assert similarity_matrix.shape == (10, 10)
assert similarity_matrix.min() >= 0
assert similarity_matrix.max() <= 1

print("\nTF-IDF baseline created successfully.")

Resume vectors: (10, 298)
Job vectors: (10, 298)
Similarity matrix: (10, 10)

Example: R001 matched against all jobs
job_id
J001    0.396
J009    0.096
J010    0.093
J002    0.081
J006    0.063
J007    0.045
J008    0.043
J005    0.021
J004    0.021
J003    0.019
Name: R001, dtype: float64

TF-IDF baseline created successfully.


In [8]:
# Cell 8: Evaluate TF-IDF against reviewed relevance labels

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from scipy.stats import spearmanr

# Convert each reviewed pair into its TF-IDF similarity score
evaluation_results = evaluation_df.copy()

evaluation_results["tfidf_score"] = evaluation_results.apply(
    lambda row: similarity_df.loc[
        row["resume_id"], row["job_id"]
    ],
    axis=1
)

# Rank jobs separately for each resume.
# Evaluate resumes that have at least one reviewed pair.
rank_metrics = []

for resume_id, group in evaluation_results.groupby("resume_id"):
    group = group.copy()

    # Higher similarity means a higher predicted rank
    group["predicted_rank"] = (
        group["tfidf_score"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    # Relevant = partial or strong match
    group["relevant"] = (
        group["relevance_label"] >= 1
    ).astype(int)

    # Precision@K: proportion of top-K reviewed jobs that are relevant
    k = min(3, len(group))
    top_k = group.nsmallest(k, "predicted_rank")
    precision_at_k = top_k["relevant"].mean()

    # Reciprocal rank: position of first relevant reviewed job
    relevant_ranks = group.loc[
        group["relevant"] == 1, "predicted_rank"
    ]

    reciprocal_rank = (
        1 / relevant_ranks.min()
        if not relevant_ranks.empty
        else 0.0
    )

    # nDCG uses the original 0/1/2 relevance labels
    ndcg = ndcg_score(
        [group["relevance_label"].to_numpy()],
        [group["tfidf_score"].to_numpy()]
    ) if len(group) > 1 else np.nan

    # Correlation between score and human relevance labels
    correlation = (
        spearmanr(
            group["tfidf_score"],
            group["relevance_label"]
        ).statistic
        if group["relevance_label"].nunique() > 1
        and group["tfidf_score"].nunique() > 1
        else np.nan
    )

    rank_metrics.append({
        "resume_id": resume_id,
        "reviewed_jobs": len(group),
        "precision_at_3": precision_at_k,
        "reciprocal_rank": reciprocal_rank,
        "ndcg": ndcg,
        "spearman_correlation": correlation
    })

metrics_df = pd.DataFrame(rank_metrics)

print("TF-IDF evaluation by resume:")
display(metrics_df.round(3))

print("\nMean metrics across evaluated resumes:")
display(
    metrics_df[
        [
            "precision_at_3",
            "reciprocal_rank",
            "ndcg",
            "spearman_correlation"
        ]
    ].mean().to_frame("mean").round(3)
)

# Save pair-level scores and evaluation metrics
results_path = EVALUATION_DIR / "tfidf_pair_scores.csv"
metrics_path = EVALUATION_DIR / "tfidf_metrics_by_resume.csv"

evaluation_results.to_csv(results_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print("\nSaved pair scores:", results_path)
print("Saved ranking metrics:", metrics_path)
print("\nTF-IDF evaluation complete.")

TF-IDF evaluation by resume:


,resume_id,reviewed_jobs,precision_at_3,reciprocal_rank,ndcg,spearman_correlation
0,R001,2,1.000,1.0,1.0,1.000
1,R002,3,0.667,1.0,1.0,0.866
2,R003,3,1.000,1.0,1.0,0.866
3,R004,2,1.000,1.0,1.0,1.000
4,R005,2,1.000,1.0,1.0,1.000
5,R006,2,0.500,1.0,1.0,1.000
6,R007,1,1.000,1.0,NaN,NaN
7,R008,1,1.000,1.0,NaN,NaN
8,R009,2,0.500,1.0,1.0,1.000
9,R010,2,1.000,1.0,1.0,1.000



Mean metrics across evaluated resumes:


,mean
precision_at_3,0.867
reciprocal_rank,1.000
ndcg,1.000
spearman_correlation,0.967



Saved pair scores: /content/talentmatch/data/evaluation/tfidf_pair_scores.csv
Saved ranking metrics: /content/talentmatch/data/evaluation/tfidf_metrics_by_resume.csv

TF-IDF evaluation complete.


In [9]:
# Cell 9: Install and load Sentence Transformer

%pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

embedding_model = SentenceTransformer(MODEL_NAME)

print("\nEmbedding model loaded:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

PyTorch version: 2.14.1+cu130
CUDA available: True


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_1016/563951183.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [10]:
# Cell 10: Generate embeddings and calculate semantic similarity

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Encode resume and job text into 384-dimensional embeddings
resume_embeddings = embedding_model.encode(
    resumes_df["resume_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

job_embeddings = embedding_model.encode(
    jobs_df["job_description"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

# Calculate similarity for all resume-job combinations
semantic_matrix = cosine_similarity(
    resume_embeddings,
    job_embeddings
)

semantic_similarity_df = pd.DataFrame(
    semantic_matrix,
    index=resumes_df["resume_id"],
    columns=jobs_df["job_id"]
)

print("Resume embedding shape:", resume_embeddings.shape)
print("Job embedding shape:", job_embeddings.shape)
print("Semantic similarity matrix:", semantic_similarity_df.shape)

print("\nExample: R001 matched against all jobs")
print(
    semantic_similarity_df.loc["R001"]
    .sort_values(ascending=False)
    .round(3)
)

assert resume_embeddings.shape == (10, 384)
assert job_embeddings.shape == (10, 384)
assert semantic_matrix.shape == (10, 10)
assert semantic_matrix.min() >= -1.001
assert semantic_matrix.max() <= 1.001

# Save semantic scores for reproducibility
semantic_scores_path = EVALUATION_DIR / "semantic_similarity_scores.csv"
semantic_similarity_df.to_csv(semantic_scores_path, index=True)

print("\nSaved:", semantic_scores_path)
print("Semantic similarity calculation passed.")

Resume embedding shape: (10, 384)
Job embedding shape: (10, 384)
Semantic similarity matrix: (10, 10)

Example: R001 matched against all jobs
job_id
J001    0.720
J009    0.658
J002    0.533
J003    0.475
J010    0.436
J004    0.396
J007    0.371
J008    0.364
J005    0.337
J006    0.320
Name: R001, dtype: float32

Saved: /content/talentmatch/data/evaluation/semantic_similarity_scores.csv
Semantic similarity calculation passed.


In [11]:
# Cell 11: Compare both models against reviewed relevance labels

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

evaluation_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Attach both models' scores to every reviewed pair
comparison_df = evaluation_df.copy()

comparison_df["tfidf_score"] = comparison_df.apply(
    lambda row: similarity_df.loc[
        row["resume_id"], row["job_id"]
    ],
    axis=1
)

comparison_df["semantic_score"] = comparison_df.apply(
    lambda row: semantic_similarity_df.loc[
        row["resume_id"], row["job_id"]
    ],
    axis=1
)

# Calculate metrics on each resume's reviewed job pairs
def evaluate_ranking(group, score_column):
    relevance = group["relevance_label"].to_numpy()
    scores = group[score_column].to_numpy()

    # Skip nDCG when there is no relevance variation
    ndcg = (
        ndcg_score([relevance], [scores])
        if len(group) > 1 and len(np.unique(relevance)) > 1
        else np.nan
    )

    # Skip correlation when either variable has no variation
    correlation = (
        spearmanr(relevance, scores).statistic
        if len(group) > 1
        and len(np.unique(relevance)) > 1
        and len(np.unique(scores)) > 1
        else np.nan
    )

    # Reciprocal rank of the first relevant job
    ranked = group.assign(score=scores).sort_values(
        "score", ascending=False
    )

    relevant_ranks = np.flatnonzero(
        ranked["relevance_label"].to_numpy() >= 1
    )

    reciprocal_rank = (
        1 / (relevant_ranks[0] + 1)
        if len(relevant_ranks) else 0.0
    )

    k = min(3, len(ranked))
    precision_at_3 = (
        ranked.head(k)["relevance_label"].ge(1).mean()
    )

    return {
        "precision_at_3": precision_at_3,
        "reciprocal_rank": reciprocal_rank,
        "ndcg": ndcg,
        "spearman_correlation": correlation
    }

rows = []

for resume_id, group in comparison_df.groupby("resume_id"):
    for model_name, score_column in [
        ("TF-IDF", "tfidf_score"),
        ("Sentence Transformer", "semantic_score")
    ]:
        metrics = evaluate_ranking(group, score_column)
        rows.append({
            "resume_id": resume_id,
            "model": model_name,
            **metrics
        })

comparison_metrics_df = pd.DataFrame(rows)

print("Metrics by resume and model:")
display(comparison_metrics_df.round(3))

print("\nMean metrics by model:")
display(
    comparison_metrics_df.groupby("model")[
        [
            "precision_at_3",
            "reciprocal_rank",
            "ndcg",
            "spearman_correlation"
        ]
    ].mean().round(3)
)

print("\nTop-3 rankings for R001:")
for model_name, score_column in [
    ("TF-IDF", "tfidf_score"),
    ("Sentence Transformer", "semantic_score")
]:
    top_jobs = (
        comparison_df[comparison_df["resume_id"] == "R001"]
        .merge(jobs_df[["job_id", "job_title"]], on="job_id")
        .sort_values(score_column, ascending=False)
        .head(3)
    )

    print(f"\n{model_name}:")
    display(top_jobs[["job_id", "job_title", score_column, "relevance_label"]])

# Save comparison artifacts
comparison_scores_path = EVALUATION_DIR / "model_comparison_pair_scores.csv"
comparison_metrics_path = EVALUATION_DIR / "model_comparison_metrics.csv"

comparison_df.to_csv(comparison_scores_path, index=False)
comparison_metrics_df.to_csv(comparison_metrics_path, index=False)

print("\nSaved:", comparison_scores_path)
print("Saved:", comparison_metrics_path)
print("Model comparison completed.")

Metrics by resume and model:


,resume_id,model,precision_at_3,reciprocal_rank,ndcg,spearman_correlation
0,R001,TF-IDF,1.000,1.0,1.0,1.000
1,R001,Sentence Transformer,1.000,1.0,1.0,1.000
2,R002,TF-IDF,0.667,1.0,1.0,0.866
3,R002,Sentence Transformer,0.667,1.0,1.0,0.866
4,R003,TF-IDF,1.000,1.0,1.0,0.866
5,R003,Sentence Transformer,1.000,1.0,1.0,0.866
6,R004,TF-IDF,1.000,1.0,1.0,1.000
7,R004,Sentence Transformer,1.000,1.0,1.0,1.000
8,R005,TF-IDF,1.000,1.0,1.0,1.000
9,R005,Sentence Transformer,1.000,1.0,1.0,1.000



Mean metrics by model:


,precision_at_3,reciprocal_rank,ndcg,spearman_correlation
model,,,,
Sentence Transformer,0.867,1.0,1.0,0.967
TF-IDF,0.867,1.0,1.0,0.967



Top-3 rankings for R001:

TF-IDF:


,job_id,job_title,tfidf_score,relevance_label
0,J001,Python Backend Developer,0.395523,2
1,J009,Junior Software Engineer,0.096477,1



Sentence Transformer:


,job_id,job_title,semantic_score,relevance_label
0,J001,Python Backend Developer,0.720214,2
1,J009,Junior Software Engineer,0.658092,1



Saved: /content/talentmatch/data/evaluation/model_comparison_pair_scores.csv
Saved: /content/talentmatch/data/evaluation/model_comparison_metrics.csv
Model comparison completed.


In [13]:
# Cell 12: Create a review file for the remaining unlabeled pairs

import pandas as pd

resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
evaluation_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Generate all possible resume-job combinations
all_pairs = pd.MultiIndex.from_product(
    [
        resumes_df["resume_id"],
        jobs_df["job_id"]
    ],
    names=["resume_id", "job_id"]
).to_frame(index=False)

# Identify pairs already labeled
labeled_pairs = evaluation_df[["resume_id", "job_id"]].drop_duplicates()

review_queue = all_pairs.merge(
    labeled_pairs,
    on=["resume_id", "job_id"],
    how="left",
    indicator=True
)

review_queue = review_queue[
    review_queue["_merge"] == "left_only"
].drop(columns="_merge")

# Attach text so each pair can be reviewed
review_queue = (
    review_queue
    .merge(resumes_df, on="resume_id")
    .merge(jobs_df, on="job_id")
)

review_queue["relevance_label"] = pd.NA
review_queue["label_reason"] = ""

review_queue = review_queue[
    [
        "resume_id",
        "job_id",
        "resume_text",
        "job_title",
        "job_description",
        "relevance_label",
        "label_reason"
    ]
]

review_queue_path = EVALUATION_DIR / "remaining_pairs_for_review.csv"
review_queue.to_csv(review_queue_path, index=False)

print("Total possible pairs:", len(all_pairs))
print("Already reviewed:", len(labeled_pairs))
print("Pairs awaiting review:", len(review_queue))
print("Saved review queue:", review_queue_path)

assert len(all_pairs) == 100
assert len(labeled_pairs) == 40
assert len(review_queue) == 60

print("\nReview queue created successfully.")
display(review_queue.head(5))

Total possible pairs: 100
Already reviewed: 20
Pairs awaiting review: 80
Saved review queue: /content/talentmatch/data/evaluation/remaining_pairs_for_review.csv

Review queue created successfully.


,resume_id,job_id,resume_text,job_title,job_description,relevance_label,label_reason
0,R001,J002,\n Python developer with experience in ...,Data Analyst,"\n Required: SQL, Python, data cleaning...",<NA>,
1,R001,J003,\n Python developer with experience in ...,Machine Learning Engineer,"\n Required: Python, scikit-learn, feat...",<NA>,
2,R001,J004,\n Python developer with experience in ...,NLP Engineer,"\n Required: Python, NLP, tokenization,...",<NA>,
3,R001,J005,\n Python developer with experience in ...,Computer Vision Engineer,"\n Required: Python, CNNs, image classi...",<NA>,
4,R001,J006,\n Python developer with experience in ...,Frontend Developer,"\n Required: HTML, CSS, JavaScript, and...",<NA>,


In [14]:
# Cell 13: Select a small review batch for broader coverage

import pandas as pd

resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
evaluation_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Load the remaining unlabeled pairs
review_queue_path = EVALUATION_DIR / "remaining_pairs_for_review.csv"
review_queue = pd.read_csv(review_queue_path)

# Select two pairs per resume for the next review batch.
# Sorting by job ID makes selection deterministic and reproducible.
review_batch = (
    review_queue
    .sort_values(["resume_id", "job_id"])
    .groupby("resume_id", group_keys=False)
    .head(2)
    .copy()
)

assert len(review_batch) == 20
assert review_batch.groupby("resume_id").size().eq(2).all()
assert review_batch["relevance_label"].isna().all()

batch_path = EVALUATION_DIR / "review_batch_20.csv"
review_batch.to_csv(batch_path, index=False)

print("Review batch size:", len(review_batch))
print("Pairs per resume:")
print(review_batch.groupby("resume_id").size())

print("\nSaved:", batch_path)
display(
    review_batch[
        ["resume_id", "job_id", "job_title", "relevance_label", "label_reason"]
    ]
)

print("\nReview batch ready. Labels remain blank.")

Review batch size: 20
Pairs per resume:
resume_id
R001    2
R002    2
R003    2
R004    2
R005    2
R006    2
R007    2
R008    2
R009    2
R010    2
dtype: int64

Saved: /content/talentmatch/data/evaluation/review_batch_20.csv


,resume_id,job_id,job_title,relevance_label,label_reason
0,R001,J002,Data Analyst,NaN,NaN
1,R001,J003,Machine Learning Engineer,NaN,NaN
8,R002,J001,Python Backend Developer,NaN,NaN
9,R002,J003,Machine Learning Engineer,NaN,NaN
15,R003,J001,Python Backend Developer,NaN,NaN
16,R003,J004,NLP Engineer,NaN,NaN
22,R004,J001,Python Backend Developer,NaN,NaN
23,R004,J002,Data Analyst,NaN,NaN
30,R005,J002,Data Analyst,NaN,NaN
31,R005,J003,Machine Learning Engineer,NaN,NaN



Review batch ready. Labels remain blank.


In [15]:
# Cell 14: Display selected pairs for manual annotation

import pandas as pd

review_batch = pd.read_csv(
    EVALUATION_DIR / "review_batch_20.csv"
)

for index, row in review_batch.iterrows():
    print("=" * 90)
    print(f"PAIR {index + 1}: {row['resume_id']} → {row['job_id']}")
    print(f"JOB TITLE: {row['job_title']}")

    print("\nRESUME:")
    print(row["resume_text"])

    print("\nJOB DESCRIPTION:")
    print(row["job_description"])

    print("\nAssign label: 0 = Low | 1 = Partial | 2 = Strong")

PAIR 1: R001 → J002
JOB TITLE: Data Analyst

RESUME:

        Python developer with experience in Python, Flask, REST APIs,
        SQL, Git, and unit testing. Built web applications and automated
        data-processing workflows. Familiar with PostgreSQL and Docker.
        

JOB DESCRIPTION:

        Required: SQL, Python, data cleaning, and exploratory data analysis.
        Preferred: pandas, Excel, Power BI, dashboard creation, and reporting.
        Responsibilities include analyzing data and communicating insights.
        

Assign label: 0 = Low | 1 = Partial | 2 = Strong
PAIR 2: R001 → J003
JOB TITLE: Machine Learning Engineer

RESUME:

        Python developer with experience in Python, Flask, REST APIs,
        SQL, Git, and unit testing. Built web applications and automated
        data-processing workflows. Familiar with PostgreSQL and Docker.
        

JOB DESCRIPTION:

        Required: Python, scikit-learn, feature engineering,
        model evaluation, and machine lea

In [16]:
# Cell 15: Save reviewed annotations and evaluate both models on 40 pairs

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

# Recommended human-reviewed labels
annotations = {
    ("R001", "J002"): (1, "Python and SQL align, but analytics-specific evidence is limited."),
    ("R001", "J003"): (1, "Python aligns, but machine learning experience is not demonstrated."),
    ("R002", "J001"): (1, "Python and SQL align, but backend API and testing evidence is missing."),
    ("R002", "J003"): (1, "Python, pandas and NumPy align, but key ML requirements are not demonstrated."),
    ("R003", "J001"): (1, "Python aligns, but backend development requirements are mostly missing."),
    ("R003", "J004"): (1, "Python aligns, but NLP and Transformer experience is not demonstrated."),
    ("R004", "J001"): (0, "No explicit Python or backend development evidence."),
    ("R004", "J002"): (0, "No explicit SQL, Python or analytics evidence."),
    ("R005", "J002"): (1, "Python and SQL align, but data analysis evidence is limited."),
    ("R005", "J003"): (1, "Python aligns, but machine learning experience is not demonstrated."),
    ("R006", "J001"): (0, "Only limited overlap through Git; backend requirements are missing."),
    ("R006", "J002"): (0, "Little demonstrated analytics alignment."),
    ("R007", "J001"): (1, "Python aligns, but backend development evidence is missing."),
    ("R007", "J002"): (1, "Python and model evaluation overlap, but analytics-specific skills are limited."),
    ("R008", "J001"): (1, "Git, Docker and deployment overlap, but Python and backend evidence is missing."),
    ("R008", "J002"): (0, "Little demonstrated analytics alignment."),
    ("R009", "J001"): (1, "SQL and documentation overlap, but Python and backend evidence is missing."),
    ("R009", "J002"): (1, "SQL, Excel, reporting and dashboards align, but Python is missing."),
    ("R010", "J002"): (1, "Python and SQL align, but analytics experience is not demonstrated."),
    ("R010", "J003"): (1, "Python fundamentals align, but machine learning experience is missing.")
}

# Load review batch
review_batch = pd.read_csv(
    EVALUATION_DIR / "review_batch_20.csv"
)

assert len(review_batch) == 20
assert len(annotations) == 20

# Apply annotations
review_batch["relevance_label"] = [
    annotations[(row.resume_id, row.job_id)][0]
    for row in review_batch.itertuples()
]

review_batch["label_reason"] = [
    annotations[(row.resume_id, row.job_id)][1]
    for row in review_batch.itertuples()
]

# Save reviewed batch
batch_path = EVALUATION_DIR / "review_batch_20.csv"
review_batch.to_csv(batch_path, index=False)

# Merge with original reviewed pairs
original = pd.read_csv(
    EVALUATION_DIR / "reviewed_pairs.csv"
)

combined = pd.concat(
    [
        original,
        review_batch[
            ["resume_id", "job_id", "relevance_label", "label_reason"]
        ]
    ],
    ignore_index=True
)

assert len(combined) == 40
assert not combined.duplicated(["resume_id", "job_id"]).any()
assert combined["relevance_label"].isin([0, 1, 2]).all()

combined_path = EVALUATION_DIR / "reviewed_pairs.csv"
combined.to_csv(combined_path, index=False)

# Evaluate both models on identical reviewed pairs
combined["tfidf_score"] = combined.apply(
    lambda r: similarity_df.loc[r["resume_id"], r["job_id"]],
    axis=1
)

combined["semantic_score"] = combined.apply(
    lambda r: semantic_similarity_df.loc[r["resume_id"], r["job_id"]],
    axis=1
)

def calculate_metrics(group, score_column):
    ranked = group.sort_values(
        score_column, ascending=False
    )

    relevance = ranked["relevance_label"].to_numpy()

    k = min(3, len(ranked))
    precision = (relevance[:k] >= 1).mean()

    relevant_positions = np.where(relevance >= 1)[0]
    reciprocal_rank = (
        1 / (relevant_positions[0] + 1)
        if len(relevant_positions) else 0
    )

    ndcg = (
        ndcg_score(
            [group["relevance_label"].to_numpy()],
            [group[score_column].to_numpy()]
        )
        if len(group) > 1 and group["relevance_label"].nunique() > 1
        else np.nan
    )

    correlation = (
        spearmanr(
            group["relevance_label"],
            group[score_column]
        ).statistic
        if len(group) > 1
        and group["relevance_label"].nunique() > 1
        and group[score_column].nunique() > 1
        else np.nan
    )

    return {
        "precision_at_3": precision,
        "reciprocal_rank": reciprocal_rank,
        "ndcg": ndcg,
        "spearman_correlation": correlation
    }

results = []

for model, score_column in [
    ("TF-IDF", "tfidf_score"),
    ("Sentence Transformer", "semantic_score")
]:
    for resume_id, group in combined.groupby("resume_id"):
        results.append({
            "model": model,
            "resume_id": resume_id,
            **calculate_metrics(group, score_column)
        })

metrics = pd.DataFrame(results)

metrics_path = EVALUATION_DIR / "model_comparison_metrics_40_pairs.csv"
scores_path = EVALUATION_DIR / "model_comparison_scores_40_pairs.csv"

metrics.to_csv(metrics_path, index=False)
combined.to_csv(scores_path, index=False)

print("Reviewed pairs:", len(combined))
print("\nLabel distribution:")
print(combined["relevance_label"].value_counts().sort_index())

print("\nMean model metrics:")
display(
    metrics.groupby("model")[
        [
            "precision_at_3",
            "reciprocal_rank",
            "ndcg",
            "spearman_correlation"
        ]
    ].mean().round(3)
)

print("\nSaved annotations:", batch_path)
print("Saved combined evaluation:", combined_path)
print("Saved metrics:", metrics_path)
print("Saved pair scores:", scores_path)

print("\nCell 15 completed successfully.")

Reviewed pairs: 40

Label distribution:
relevance_label
0     8
1    21
2    11
Name: count, dtype: int64

Mean model metrics:


,precision_at_3,reciprocal_rank,ndcg,spearman_correlation
model,,,,
Sentence Transformer,0.867,1.0,1.0,0.852
TF-IDF,0.867,1.0,1.0,0.861



Saved annotations: /content/talentmatch/data/evaluation/review_batch_20.csv
Saved combined evaluation: /content/talentmatch/data/evaluation/reviewed_pairs.csv
Saved metrics: /content/talentmatch/data/evaluation/model_comparison_metrics_40_pairs.csv
Saved pair scores: /content/talentmatch/data/evaluation/model_comparison_scores_40_pairs.csv

Cell 15 completed successfully.


In [17]:
import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

print("--- Milestone 3: Evidence-Based Skill Matching Implementation ---\n")

# 1. Define transparent, configurable skill dictionaries for mapping terms to canonical skills
SKILL_PATTERNS = {
    'python': r'\bpython\b',
    'flask': r'\bflask\b',
    'fastapi': r'\bfastapi\b',
    'rest apis': r'\brest\s+apis?\b|\bapi\s+endpoints?\b|\bapis?\b',
    'sql': r'\bsql\b|\bpostgresql\b',
    'git': r'\bgit\b',
    'automated testing': r'\b(?:automated\s+)?testing\b|\bunit\s+testing\b|\bdebugging\b',
    'postgresql': r'\bpostgresql\b',
    'docker': r'\bdocker\b',
    'pandas': r'\bpandas\b',
    'numpy': r'\bnumpy\b',
    'excel': r'\bexcel\b',
    'power bi': r'\bpower\s+bi\b',
    'exploratory data analysis': r'\bexploratory\s+data\s+analysis\b|\bdata\s+cleaning\b|\banalyzing\s+data\b',
    'data cleaning': r'\bdata\s+cleaning\b',
    'reporting': r'\breporting\b|\breports?\b',
    'dashboard creation': r'\bdashboard\b|\bdashboards?\b',
    'scikit-learn': r'\bscikit-learn\b|\bmodel\s+evaluation\b',
    'feature engineering': r'\bfeature\s+engineering\b',
    'model evaluation': r'\bmodel\s+evaluation\b',
    'machine learning': r'\bmachine\s+learning\b|\bml\b',
    'cross-validation': r'\bcross-validation\b',
    'classification': r'\bclassification\b',
    'regression': r'\bregression\b',
    'transformers': r'\btransformers\b',
    'tokenization': r'\btokenization\b',
    'text classification': r'\btext\s+classification\b',
    'sentence embeddings': r'\bsentence\s+embeddings?\b',
    'semantic search': r'\bsemantic\s+search\b',
    'hugging face': r'\bhugging\s+face\b',
    'pytorch': r'\bpytorch\b',
    'nlp': r'\bnlp\b|\btext-processing\b',
    'html': r'\bhtml\b',
    'css': r'\bcss\b',
    'javascript': r'\bjavascript\b',
    'react': r'\breact\b',
    'responsive web design': r'\bresponsive\b',
    'accessibility': r'\baccessibility\b',
    'linux': r'\blinux\b',
    'ci/cd': r'\bci/cd\b',
    'deployment automation': r'\bdeployment\s+automation\b|\bdeployment\b',
    'aws': r'\baws\b',
    'monitoring': r'\bmonitoring\b',
    'container orchestration': r'\bcontainer\b',
    'reporting': r'\breporting\b',
    'stakeholder communication': r'\bstakeholder\b',
    'requirements gathering': r'\brequirements\b',
    'programming fundamentals': r'\bprogramming\s+fundamentals\b|\bobject-oriented\b',
    'data structures': r'\bdata\s+structures\b'
}

def extract_skills(text):
    matched_skills = set()
    clean_text = str(text).lower()
    for skill, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, clean_text):
            matched_skills.add(skill)
    return matched_skills

# Job explicit requirements (extracted statically based on original prompt descriptions)
JOB_REQUIREMENTS = {
    'J001': {'required': {'python', 'rest apis', 'sql', 'git', 'automated testing'}, 'preferred': {'fastapi', 'postgresql', 'docker', 'deployment automation'}},
    'J002': {'required': {'sql', 'python', 'data cleaning', 'exploratory data analysis'}, 'preferred': {'pandas', 'excel', 'power bi', 'dashboard creation', 'reporting'}},
    'J003': {'required': {'python', 'scikit-learn', 'feature engineering', 'model evaluation', 'machine learning'}, 'preferred': {'pandas', 'numpy', 'cross-validation', 'classification', 'regression'}},
    'J004': {'required': {'python', 'nlp', 'tokenization', 'transformers', 'text classification'}, 'preferred': {'hugging face', 'pytorch', 'sentence embeddings', 'semantic search'}},
    'J005': {'required': {'python', 'transformers', 'classification', 'pytorch'}, 'preferred': {'scikit-learn', 'model evaluation'}}, # CV metrics matched to model evaluation
    'J006': {'required': {'html', 'css', 'javascript', 'responsive web design'}, 'preferred': {'react', 'accessibility', 'git'}},
    'J007': {'required': {'linux', 'docker', 'ci/cd', 'deployment automation'}, 'preferred': {'aws', 'git', 'monitoring', 'container orchestration'}},
    'J008': {'required': {'sql', 'excel', 'reporting', 'exploratory data analysis'}, 'preferred': {'power bi', 'stakeholder communication', 'requirements gathering', 'dashboard creation'}},
    'J009': {'required': {'programming fundamentals', 'data structures', 'automated testing', 'git'}, 'preferred': {'python', 'sql'}},
    'J010': {'required': {'python', 'machine learning', 'model evaluation'}, 'preferred': {'pytorch', 'transformers', 'nlp', 'sentence embeddings', 'semantic search'}}
}

# Load original source documents and the latest reviewed pairs
resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
reviewed_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Apply skill extraction to resumes
resumes_df['extracted_skills'] = resumes_df['resume_text'].apply(extract_skills)
resume_skills_map = dict(zip(resumes_df['resume_id'], resumes_df['extracted_skills']))

# 2. Compute matching metrics for each reviewed pair
skill_matching_rows = []
for idx, row in reviewed_df.iterrows():
    r_id, j_id = row['resume_id'], row['job_id']
    r_skills = resume_skills_map.get(r_id, set())

    reqs = JOB_REQUIREMENTS.get(j_id, {'required': set(), 'preferred': set()})
    req_skills = reqs['required']
    pref_skills = reqs['preferred']

    matched_req = r_skills.intersection(req_skills)
    missing_req = req_skills - r_skills
    matched_pref = r_skills.intersection(pref_skills)

    coverage = len(matched_req) / len(req_skills) if req_skills else 0.0

    # Retrieve semantic similarity from baseline
    sem_score = semantic_similarity_df.loc[r_id, j_id]

    # Explanation generation emphasizing presence vs absence without claiming lack of ability
    matched_req_str = ", ".join(matched_req) if matched_req else "None"
    missing_req_str = ", ".join(missing_req) if missing_req else "None"
    matched_pref_str = ", ".join(matched_pref) if matched_pref else "None"

    explanation = (
        f"Demonstrated alignment includes required skills: [{matched_req_str}] "
        f"and preferred skills: [{matched_pref_str}]. "
        f"The following required skills were not explicitly stated in the resume: [{missing_req_str}]."
    )

    skill_matching_rows.append({
        'resume_id': r_id,
        'job_id': j_id,
        'matched_required': list(matched_req),
        'missing_required': list(missing_req),
        'matched_preferred': list(matched_pref),
        'required_coverage': coverage,
        'semantic_score': sem_score,
        'evidence_explanation': explanation
    })

skill_matching_df = pd.DataFrame(skill_matching_rows)

# 3. Create configurable hybrid score combining semantic similarity and explicit skill matching
# Weighting: 50% semantic similarity + 50% required skill coverage
SEMANTIC_WEIGHT = 0.5
COVERAGE_WEIGHT = 0.5

skill_matching_df['hybrid_score'] = (
    (skill_matching_df['semantic_score'] * SEMANTIC_WEIGHT) +
    (skill_matching_df['required_coverage'] * COVERAGE_WEIGHT)
)

# Save skill matching results
sm_path = EVALUATION_DIR / "skill_matching_results.csv"
skill_matching_df.to_csv(sm_path, index=False)
print(f"Saved Skill Matching Results to: {sm_path}")

# 4. Rank jobs and construct evaluation comparison metrics
def evaluate_model_metrics(df, rank_column):
    results = []
    for resume_id, group in df.groupby("resume_id"):
        relevance = group["relevance_label"].to_numpy()
        scores = group[rank_column].to_numpy()

        ndcg = (
            ndcg_score([relevance], [scores])
            if len(group) > 1 and len(np.unique(relevance)) > 1
            else np.nan
        )

        correlation = (
            spearmanr(relevance, scores).statistic
            if len(group) > 1
            and len(np.unique(relevance)) > 1
            and len(np.unique(scores)) > 1
            else np.nan
        )

        ranked = group.sort_values(rank_column, ascending=False)
        k = min(3, len(ranked))
        precision_at_3 = (ranked.head(k)["relevance_label"].ge(1).mean())

        results.append({
            "resume_id": resume_id,
            "precision_at_3": precision_at_3,
            "ndcg": ndcg,
            "spearman_correlation": correlation
        })
    return pd.DataFrame(results)

# Add the relevance labels back for ranking evaluation
hybrid_ranking_df = skill_matching_df.merge(
    reviewed_df[['resume_id', 'job_id', 'relevance_label']],
    on=['resume_id', 'job_id']
)

hybrid_metrics = evaluate_model_metrics(hybrid_ranking_df, 'hybrid_score')
semantic_metrics = evaluate_model_metrics(hybrid_ranking_df, 'semantic_score')

hybrid_metrics['model'] = 'Hybrid (Semantic + Coverage)'
semantic_metrics['model'] = 'Sentence Transformer (Semantic-Only)'

comparison_summary = pd.concat([hybrid_metrics, semantic_metrics], ignore_index=True)

# Save ranking results
hy_ranking_path = EVALUATION_DIR / "hybrid_ranking_results.csv"
hybrid_ranking_df.to_csv(hy_ranking_path, index=False)
print(f"Saved Hybrid Ranking Results to: {hy_ranking_path}")

# 5. Milestone 3 Validation Checks
assert not skill_matching_df.isna().all().any(), "Detected unexpected null values in results."
assert not skill_matching_df.duplicated(['resume_id', 'job_id']).any(), "Detected duplicate evaluation pairs."
assert skill_matching_df['hybrid_score'].min() >= 0.0 and skill_matching_df['hybrid_score'].max() <= 1.0, "Hybrid scores must stay within [0, 1]."

print("\n--- Mean Performance Metrics Comparison ---")
display(comparison_summary.groupby('model')[['precision_at_3', 'ndcg', 'spearman_correlation']].mean().round(3))

print("\n--- Sample Hybrid Recommendations Preview ---")
display(hybrid_ranking_df[['resume_id', 'job_id', 'required_coverage', 'semantic_score', 'hybrid_score', 'relevance_label']].head(8))


--- Milestone 3: Evidence-Based Skill Matching Implementation ---

Saved Skill Matching Results to: /content/talentmatch/data/evaluation/skill_matching_results.csv
Saved Hybrid Ranking Results to: /content/talentmatch/data/evaluation/hybrid_ranking_results.csv

--- Mean Performance Metrics Comparison ---


,precision_at_3,ndcg,spearman_correlation
model,,,
Hybrid (Semantic + Coverage),0.867,1.0,0.852
Sentence Transformer (Semantic-Only),0.867,1.0,0.852



--- Sample Hybrid Recommendations Preview ---


,resume_id,job_id,required_coverage,semantic_score,hybrid_score,relevance_label
0,R001,J001,1.00,0.720214,0.860107,2
1,R002,J002,1.00,0.713044,0.856522,2
2,R003,J003,1.00,0.766207,0.883104,2
3,R004,J004,0.80,0.750067,0.775034,2
4,R007,J005,0.75,0.714518,0.732259,2
5,R006,J006,1.00,0.617792,0.808896,2
6,R008,J007,1.00,0.680638,0.840319,2
7,R009,J008,0.75,0.548826,0.649413,2


In [18]:
import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

print("--- Milestone 3.1: Correct Evidence-Based Skill Matching Implementation ---\n")

# 1. Precise, non-overlapping skill patterns with strict word boundaries
CORRECTED_SKILL_PATTERNS = {
    'python': r'\bpython\b',
    'flask': r'\bflask\b',
    'fastapi': r'\bfastapi\b',
    'rest apis': r'\brest\s+apis?\b|\bapi\s+endpoints?\b',
    'sql': r'\bsql\b',
    'postgresql': r'\bpostgresql\b',
    'git': r'\bgit\b',
    'automated testing': r'\b(?:automated\s+)?testing\b|\bunit\s+testing\b|\bdebugging\b',
    'docker': r'\bdocker\b',
    'pandas': r'\bpandas\b',
    'numpy': r'\bnumpy\b',
    'excel': r'\bexcel\b',
    'power bi': r'\bpower\s+bi\b',
    'exploratory data analysis': r'\bexploratory\s+data\s+analysis\b',
    'data cleaning': r'\bdata\s+cleaning\b',
    'reporting': r'\breporting\b|\breports?\b',
    'dashboard creation': r'\bdashboard\b|\bdashboards?\b|\bdashboard\s+development\b',
    'scikit-learn': r'\bscikit-learn\b',
    'feature engineering': r'\bfeature\s+engineering\b',
    'model evaluation': r'\bmodel\s+evaluation\b',
    'machine learning': r'\bmachine\s+learning\b|\bml\b',
    'cross-validation': r'\bcross-validation\b',
    'classification': r'\bclassification\b',
    'regression': r'\bregression\b',
    'transformers': r'\btransformers\b',
    'tokenization': r'\btokenization\b',
    'text classification': r'\btext\s+classification\b',
    'sentence embeddings': r'\bsentence\s+embeddings?\b',
    'semantic search': r'\bsemantic\s+search\b',
    'hugging face': r'\bhugging\s+face\b',
    'pytorch': r'\bpytorch\b',
    'nlp': r'\bnlp\b|\btext-processing\b',
    'html': r'\bhtml\b',
    'css': r'\bcss\b',
    'javascript': r'\bjavascript\b',
    'react': r'\breact\b',
    'responsive web design': r'\bresponsive\b',
    'accessibility': r'\baccessibility\b',
    'linux': r'\blinux\b',
    'ci/cd': r'\bci/cd\b',
    'deployment automation': r'\bdeployment\s+automation\b|\bdeployment\b',
    'aws': r'\baws\b',
    'monitoring': r'\bmonitoring\b',
    'container orchestration': r'\bcontainer\b',
    'stakeholder communication': r'\bstakeholder\b',
    'requirements gathering': r'\brequirements\b',
    'programming fundamentals': r'\bprogramming\s+fundamentals\b|\bobject-oriented\b',
    'data structures': r'\bdata\s+structures\b',
    'cnns': r'\bcnns?\b',
    'image classification': r'\bimage\s+classification\b',
    'opencv': r'\bopencv\b',
    'transfer learning': r'\btransfer\s+learning\b',
    'image augmentation': r'\bimage\s+augmentation\b'
}

def extract_skills_corrected(text):
    matched_skills = set()
    clean_text = str(text).lower()
    for skill, pattern in CORRECTED_SKILL_PATTERNS.items():
        if re.search(pattern, clean_text):
            matched_skills.add(skill)
    return matched_skills

# 2. Corrected mappings reflecting source job description texts strictly
# Note: J005 is correctly set to CV requirements rather than NLP/Transformers
CORRECTED_JOB_REQUIREMENTS = {
    'J001': {'required': {'python', 'rest apis', 'sql', 'git', 'automated testing'}, 'preferred': {'fastapi', 'postgresql', 'docker', 'deployment automation'}},
    'J002': {'required': {'sql', 'python', 'data cleaning', 'exploratory data analysis'}, 'preferred': {'pandas', 'excel', 'power bi', 'dashboard creation', 'reporting'}},
    'J003': {'required': {'python', 'scikit-learn', 'feature engineering', 'model evaluation', 'machine learning'}, 'preferred': {'pandas', 'numpy', 'cross-validation', 'classification', 'regression'}},
    'J004': {'required': {'python', 'nlp', 'tokenization', 'transformers', 'text classification'}, 'preferred': {'hugging face', 'pytorch', 'sentence embeddings', 'semantic search'}},
    'J005': {'required': {'python', 'cnns', 'image classification', 'pytorch'}, 'preferred': {'opencv', 'image augmentation', 'transfer learning', 'model evaluation'}},
    'J006': {'required': {'html', 'css', 'javascript', 'responsive web design'}, 'preferred': {'react', 'accessibility', 'git'}},
    'J007': {'required': {'linux', 'docker', 'ci/cd', 'deployment automation'}, 'preferred': {'aws', 'git', 'monitoring', 'container orchestration'}},
    'J008': {'required': {'sql', 'excel', 'reporting', 'exploratory data analysis'}, 'preferred': {'power bi', 'stakeholder communication', 'requirements gathering', 'dashboard creation'}},
    'J009': {'required': {'programming fundamentals', 'data structures', 'automated testing', 'git'}, 'preferred': {'python', 'sql'}},
    'J010': {'required': {'python', 'machine learning', 'model evaluation'}, 'preferred': {'pytorch', 'transformers', 'nlp', 'sentence embeddings', 'semantic search'}}
}

# Load datasets without mutating historical inputs
resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
reviewed_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Re-extract skills utilizing the clean rules
resumes_df['extracted_skills'] = resumes_df['resume_text'].apply(extract_skills_corrected)
resume_skills_map = dict(zip(resumes_df['resume_id'], resumes_df['extracted_skills']))

# 3. Match metrics for the reviewed pairs
corrected_matching_rows = []
for idx, row in reviewed_df.iterrows():
    r_id, j_id = row['resume_id'], row['job_id']
    r_skills = resume_skills_map.get(r_id, set())

    reqs = CORRECTED_JOB_REQUIREMENTS.get(j_id, {'required': set(), 'preferred': set()})
    req_skills = reqs['required']
    pref_skills = reqs['preferred']

    matched_req = r_skills.intersection(req_skills)
    missing_req = req_skills - r_skills
    matched_pref = r_skills.intersection(pref_skills)

    coverage = len(matched_req) / len(req_skills) if req_skills else 0.0

    sem_score = semantic_similarity_df.loc[r_id, j_id]

    # Generate factual and clean evidence explanations
    matched_req_str = ", ".join(sorted(matched_req)) if matched_req else "None"
    missing_req_str = ", ".join(sorted(missing_req)) if missing_req else "None"
    matched_pref_str = ", ".join(sorted(matched_pref)) if matched_pref else "None"

    explanation = (
        f"Demonstrated alignment includes required skills: [{matched_req_str}] "
        f"and preferred skills: [{matched_pref_str}]. "
        f"Required skills not explicitly stated: [{missing_req_str}]."
    )

    corrected_matching_rows.append({
        'resume_id': r_id,
        'job_id': j_id,
        'matched_required': list(matched_req),
        'missing_required': list(missing_req),
        'matched_preferred': list(matched_pref),
        'required_coverage': coverage,
        'semantic_score': sem_score,
        'evidence_explanation': explanation
    })

corrected_matching_df = pd.DataFrame(corrected_matching_rows)

# Configure and assign hybrid weights
SEMANTIC_WEIGHT = 0.5
COVERAGE_WEIGHT = 0.5

corrected_matching_df['hybrid_score'] = (
    (corrected_matching_df['semantic_score'] * SEMANTIC_WEIGHT) +
    (corrected_matching_df['required_coverage'] * COVERAGE_WEIGHT)
)

# Save skill matching results
sm_path = EVALUATION_DIR / "skill_matching_results.csv"
corrected_matching_df.to_csv(sm_path, index=False)
print(f"Saved Corrected Skill Matching Results to: {sm_path}")

# Add the relevance labels back for ranking evaluations
hybrid_ranking_df = corrected_matching_df.merge(
    reviewed_df[['resume_id', 'job_id', 'relevance_label']],
    on=['resume_id', 'job_id']
)

# Save hybrid ranking results
hy_ranking_path = EVALUATION_DIR / "hybrid_ranking_results.csv"
hybrid_ranking_df.to_csv(hy_ranking_path, index=False)
print(f"Saved Corrected Hybrid Ranking Results to: {hy_ranking_path}")

# 4. Strict Validation Checks
assert not corrected_matching_df.isna().any().any(), "Detected null values in outputs!"
assert not corrected_matching_df.duplicated(['resume_id', 'job_id']).any(), "Duplicate pairs present!"
assert len(corrected_matching_df) == 40, "Expected all 40 annotated pairs to be evaluated."
assert corrected_matching_df['hybrid_score'].min() >= 0.0 and corrected_matching_df['hybrid_score'].max() <= 1.0, "Hybrid scores outside range."
for j_id, reqs in CORRECTED_JOB_REQUIREMENTS.items():
    assert len(reqs['required']) > 0, f"Job {j_id} is missing required skills."

# Verify coverage math matches matched_required counts
for idx, row in corrected_matching_df.iterrows():
    reqs = CORRECTED_JOB_REQUIREMENTS[row['job_id']]['required']
    expected_cov = len(row['matched_required']) / len(reqs)
    assert abs(row['required_coverage'] - expected_cov) < 1e-7, "Required coverage mismatch!"

print("\nAll validation checks successfully completed!")

# 5. Fair Evaluation Comparisons
def evaluate_metrics(df, score_col):
    results = []
    for r_id, group in df.groupby("resume_id"):
        relevance = group["relevance_label"].to_numpy()
        scores = group[score_col].to_numpy()

        # Sort descending by predictions to calculate metrics
        ranked = group.sort_values(score_col, ascending=False)
        rel_sorted = ranked["relevance_label"].to_numpy()

        # Precision@3
        k = min(3, len(ranked))
        p_at_k = (rel_sorted[:k] >= 1).mean() if k > 0 else 0.0

        # MRR
        rel_indices = np.where(rel_sorted >= 1)[0]
        mrr = 1 / (rel_indices[0] + 1) if len(rel_indices) > 0 else 0.0

        # nDCG
        ndcg = (
            ndcg_score([relevance], [scores])
            if len(group) > 1 and len(np.unique(relevance)) > 1
            else np.nan
        )

        # Spearman Correlation
        corr = (
            spearmanr(relevance, scores).statistic
            if len(group) > 1 and len(np.unique(relevance)) > 1 and len(np.unique(scores)) > 1
            else np.nan
        )

        results.append({
            "resume_id": r_id,
            "precision_at_3": p_at_k,
            "mrr": mrr,
            "ndcg": ndcg,
            "spearman_correlation": corr
        })
    return pd.DataFrame(results)

hybrid_eval = evaluate_metrics(hybrid_ranking_df, 'hybrid_score')
semantic_eval = evaluate_metrics(hybrid_ranking_df, 'semantic_score')

hybrid_eval['model'] = 'Hybrid (Semantic + Coverage)'
semantic_eval['model'] = 'Sentence Transformer (Semantic-Only)'

comparison_summary_corrected = pd.concat([hybrid_eval, semantic_eval], ignore_index=True)

# Save corrected metrics summary
metrics_corrected_path = EVALUATION_DIR / "model_comparison_metrics_corrected.csv"
comparison_summary_corrected.to_csv(metrics_corrected_path, index=False)
print(f"Saved Corrected Model Comparison to: {metrics_corrected_path}")

print("\n--- Mean Performance Metrics Comparison (N=40 reviewed pairs) ---")
display(comparison_summary_corrected.groupby('model')[['precision_at_3', 'mrr', 'ndcg', 'spearman_correlation']].mean().round(3))


--- Milestone 3.1: Correct Evidence-Based Skill Matching Implementation ---

Saved Corrected Skill Matching Results to: /content/talentmatch/data/evaluation/skill_matching_results.csv
Saved Corrected Hybrid Ranking Results to: /content/talentmatch/data/evaluation/hybrid_ranking_results.csv

All validation checks successfully completed!
Saved Corrected Model Comparison to: /content/talentmatch/data/evaluation/model_comparison_metrics_corrected.csv

--- Mean Performance Metrics Comparison (N=40 reviewed pairs) ---


,precision_at_3,mrr,ndcg,spearman_correlation
model,,,,
Hybrid (Semantic + Coverage),0.867,1.0,1.0,0.852
Sentence Transformer (Semantic-Only),0.867,1.0,1.0,0.852


In [ ]:
import pandas as pd
import numpy as np
import re

# Load verified source data
resumes_df = pd.read_csv(SYNTHETIC_DIR / 'resumes.csv')
jobs_df = pd.read_csv(SYNTHETIC_DIR / 'job_descriptions.csv')
hybrid_ranking_df = pd.read_csv(EVALUATION_DIR / 'hybrid_ranking_results.csv')

print("=== STEP 1: DISPLAY SOURCE REQUIREMENTS ===\n")
for idx, row in jobs_df.iterrows():
    print(f"Job ID: {row['job_id']} | Title: {row['job_title']}")
    print(f"Description:\n{row['job_description'].strip()}")
    print("-" * 80)

print("\n=== STEP 2: AUDIT SKILL MAPPINGS ===\n")
# Programmatic audit comparison against the current mapping dictionary
mapping_audit = []
for idx, row in jobs_df.iterrows():
    j_id = row['job_id']
    current_map = CORRECTED_JOB_REQUIREMENTS.get(j_id, {'required': set(), 'preferred': set()})

    # Manual extraction reference from natural text
    # J001: Required: Python, REST APIs, SQL, Git, and automated testing. Preferred: FastAPI, PostgreSQL, Docker, application deployment
    # J002: Required: SQL, Python, data cleaning, exploratory data analysis. Preferred: pandas, Excel, Power BI, dashboard creation, reporting
    # J003: Required: Python, scikit-learn, feature engineering, model evaluation, machine learning. Preferred: pandas, NumPy, cross-validation, classification, regression
    # J004: Required: Python, NLP, tokenization, Transformers, text classification. Preferred: Hugging Face, PyTorch, sentence embeddings, semantic search
    # J005: Required: Python, CNNs, image classification, PyTorch. Preferred: OpenCV, image augmentation, transfer learning, model evaluation
    # J006: Required: HTML, CSS, JavaScript, responsive web design. Preferred: React, accessibility, Git, reusable UI components
    # J007: Required: Linux, Docker, CI/CD, deployment automation. Preferred: AWS, Git, monitoring, container orchestration
    # J008: Required: SQL, Excel, reporting, data analysis. Preferred: Power BI, stakeholder communication, requirements gathering, dashboard development
    # J009: Required: programming fundamentals, data structures, debugging, Git, software testing. Preferred: Python, Java, SQL, object-oriented programming
    # J010: Required: Python, machine learning, model evaluation, AI applications. Preferred: PyTorch, Transformers, NLP, sentence embeddings, semantic search

    print(f"Job: {j_id} - {row['job_title']}")
    print(f"  Current Map Required : {sorted(list(current_map['required']))}")
    print(f"  Current Map Preferred: {sorted(list(current_map['preferred']))}")
    print("")

print("\n=== STEP 3 & 4: REGEX RISKS AND ALIAS POLICY ===\n")
regex_risks = [
    ("automated testing vs debugging", "\\b(?:automated\\s+)?testing\\b matches testing and debugging loosely if not isolated."),
    ("responsive design vs responsive", "\\bresponsive\\b pattern matches responsive design but might catch general layout comments."),
    ("deployment automation vs deployment", "\\bdeployment\\b matches deployment automation broadly without specifying automation pipelines."),
    ("container orchestration vs container", "\\bcontainer\\b matches docker and local containers rather than orchestration (e.g. k8s)."),
    ("stakeholder communication vs stakeholder", "\\bstakeholder\\b matches any mention of stakeholder."),
    ("requirements gathering vs requirements", "\\brequirements\\b matches job description titles or section headings rather than active elicitation.")
]
for term, desc in regex_risks:
    print(f"- Risk [{term}]: {desc}")

print("\nAlias Policy Recommendation:")
print("PostgreSQL matches should fall back to general SQL evidence without double-counting as a required skill unless explicitly written in job requirements.")

print("\n=== STEP 5: RANKING VALIDATION & EXPLICIT ASSERTIONS ===\n")
# Programmatic validation of sorting order descending for every single resume
failures = 0
for r_id, group in hybrid_ranking_df.groupby("resume_id"):
    scores = group["hybrid_score"].to_numpy()
    is_sorted_descending = np.all(np.diff(scores) <= 0)
    # If not fully sorted descending, raise assertion
    assert is_sorted_descending, f"Assertion Failed: Resume {r_id} is not sorted descending! Scores: {scores}"
    print(f"Resume {r_id}: Sorted descending? Yes. Scores: {scores.round(3)}")

print("\nAll ranking validation assertions successfully passed.")

In [20]:
import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

print("=== Milestone 3.3: Final Matching Engine Correction ===\n")

# 1. Definitive Non-overlapping Skill Patterns with Strict Word Boundaries
FINAL_SKILL_PATTERNS = {
    'python': r'\bpython\b',
    'flask': r'\bflask\b',
    'fastapi': r'\bfastapi\b',
    'rest apis': r'\brest\s+apis?\b|\bapi\s+endpoints?\b',
    'sql': r'\bsql\b',
    'postgresql': r'\bpostgresql\b',
    'git': r'\bgit\b',
    'automated testing': r'\bautomated\s+testing\b',
    'software testing': r'\bsoftware\s+testing\b',
    'debugging': r'\bdebugging\b',
    'docker': r'\bdocker\b',
    'pandas': r'\bpandas\b',
    'numpy': r'\bnumpy\b',
    'excel': r'\bexcel\b',
    'power bi': r'\bpower\s+bi\b',
    'exploratory data analysis': r'\bexploratory\s+data\s+analysis\b',
    'data cleaning': r'\bdata\s+cleaning\b',
    'data analysis': r'\bdata\s+analysis\b',
    'reporting': r'\breporting\b|\breports?\b',
    'dashboard creation': r'\bdashboard\s+creation\b|\bdashboards?\b|\bdashboard\s+development\b',
    'scikit-learn': r'\bscikit-learn\b',
    'feature engineering': r'\bfeature\s+engineering\b',
    'model evaluation': r'\bmodel\s+evaluation\b',
    'machine learning': r'\bmachine\s+learning\b|\bml\b',
    'cross-validation': r'\bcross-validation\b',
    'classification': r'\bclassification\b',
    'regression': r'\bregression\b',
    'transformers': r'\btransformers\b',
    'tokenization': r'\btokenization\b',
    'text classification': r'\btext\s+classification\b',
    'sentence embeddings': r'\bsentence\s+embeddings?\b',
    'semantic search': r'\bsemantic\s+search\b',
    'hugging face': r'\bhugging\s+face\b',
    'pytorch': r'\bpytorch\b',
    'nlp': r'\bnlp\b|\btext-processing\b',
    'html': r'\bhtml\b',
    'css': r'\bcss\b',
    'javascript': r'\bjavascript\b',
    'react': r'\breact\b',
    'responsive web design': r'\bresponsive\s+(?:web\s+)?design\b',
    'accessibility': r'\baccessibility\b',
    'linux': r'\blinux\b',
    'ci/cd': r'\bci/cd\b',
    'deployment': r'\bdeployment\b',
    'deployment automation': r'\bdeployment\s+automation\b',
    'aws': r'\baws\b',
    'monitoring': r'\bmonitoring\b',
    'containerization': r'\bcontainerized\b|\bcontainers?\b',
    'container orchestration': r'\bcontainer\s+orchestration\b',
    'stakeholder communication': r'\bstakeholder\s+communication\b|\bstakeholder\b',
    'requirements gathering': r'\brequirements\s+gathering\b',
    'programming fundamentals': r'\bprogramming\s+fundamentals\b',
    'data structures': r'\bdata\s+structures\b',
    'cnns': r'\bcnns?\b',
    'image classification': r'\bimage\s+classification\b',
    'opencv': r'\bopencv\b',
    'transfer learning': r'\btransfer\s+learning\b',
    'image augmentation': r'\bimage\s+augmentation\b',
    'ai applications': r'\bai\s+applications?\b'
}

def extract_skills_final(text):
    matched_skills = set()
    clean_text = str(text).lower()
    for skill, pattern in FINAL_SKILL_PATTERNS.items():
        if re.search(pattern, clean_text):
            matched_skills.add(skill)
    return matched_skills

# 2. Definitively Audited Requirements from Source Texts
FINAL_JOB_REQUIREMENTS = {
    'J001': {'required': {'python', 'rest apis', 'sql', 'git', 'automated testing'}, 'preferred': {'fastapi', 'postgresql', 'docker', 'deployment'}},
    'J002': {'required': {'sql', 'python', 'data cleaning', 'exploratory data analysis'}, 'preferred': {'pandas', 'excel', 'power bi', 'dashboard creation', 'reporting'}},
    'J003': {'required': {'python', 'scikit-learn', 'feature engineering', 'model evaluation', 'machine learning'}, 'preferred': {'pandas', 'numpy', 'cross-validation', 'classification', 'regression'}},
    'J004': {'required': {'python', 'nlp', 'tokenization', 'transformers', 'text classification'}, 'preferred': {'hugging face', 'pytorch', 'sentence embeddings', 'semantic search'}},
    'J005': {'required': {'python', 'cnns', 'image classification', 'pytorch'}, 'preferred': {'opencv', 'image augmentation', 'transfer learning', 'model evaluation'}},
    'J006': {'required': {'html', 'css', 'javascript', 'responsive web design'}, 'preferred': {'react', 'accessibility', 'git'}},
    'J007': {'required': {'linux', 'docker', 'ci/cd', 'deployment automation'}, 'preferred': {'aws', 'git', 'monitoring', 'container orchestration'}},
    'J008': {'required': {'sql', 'excel', 'reporting', 'data analysis'}, 'preferred': {'power bi', 'stakeholder communication', 'requirements gathering', 'dashboard creation'}},
    'J009': {'required': {'programming fundamentals', 'data structures', 'debugging', 'git', 'software testing'}, 'preferred': {'python', 'sql'}},
    'J010': {'required': {'python', 'machine learning', 'model evaluation', 'ai applications'}, 'preferred': {'pytorch', 'transformers', 'nlp', 'sentence embeddings', 'semantic search'}}
}

# Load verified historical documents
resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
reviewed_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

# Build dictionary mapping resume ID to its exact extracted skills
resume_skills = {}
for idx, row in resumes_df.iterrows():
    resume_skills[row['resume_id']] = extract_skills_final(row['resume_text'])

# 3. Match Metrics Construction & Factual Alias Evaluation
results_rows = []
for idx, row in reviewed_df.iterrows():
    r_id, j_id = row['resume_id'], row['job_id']

    raw_skills = resume_skills.get(r_id, set())
    reqs = FINAL_JOB_REQUIREMENTS.get(j_id, {'required': set(), 'preferred': set()})
    req_skills = reqs['required']
    pref_skills = reqs['preferred']

    # Strict Alias Logic Implementation
    # If resume contains 'postgresql', map it as evidence for general 'sql'
    effective_skills = raw_skills.copy()
    if 'postgresql' in effective_skills:
        effective_skills.add('sql')

    # Intersections
    matched_req = effective_skills.intersection(req_skills)
    missing_req = req_skills - effective_skills
    matched_pref = effective_skills.intersection(pref_skills)

    coverage = len(matched_req) / len(req_skills) if req_skills else 0.0
    sem_score = float(semantic_similarity_df.loc[r_id, j_id])
    hybrid_score = 0.5 * sem_score + 0.5 * coverage

    matched_req_str = ", ".join(sorted(matched_req)) if matched_req else "None"
    missing_req_str = ", ".join(sorted(missing_req)) if missing_req else "None"
    matched_pref_str = ", ".join(sorted(matched_pref)) if matched_pref else "None"

    explanation = (
        f"Demonstrated alignment includes required skills: [{matched_req_str}] "
        f"and preferred skills: [{matched_pref_str}]. "
        f"Required skills not explicitly stated: [{missing_req_str}]."
    )

    results_rows.append({
        'resume_id': r_id,
        'job_id': j_id,
        'matched_required': list(matched_req),
        'missing_required': list(missing_req),
        'matched_preferred': list(matched_pref),
        'required_coverage': coverage,
        'semantic_score': sem_score,
        'hybrid_score': hybrid_score,
        'evidence_explanation': explanation
    })

matching_results_df = pd.DataFrame(results_rows)

# 4. Save Base Skill Matching Output
sm_path = EVALUATION_DIR / "skill_matching_results.csv"
matching_results_df.to_csv(sm_path, index=False)
print(f"[SAVED] Base Skill Matching Results: {sm_path}")

# Add human relevance label for ranking evaluations
hybrid_ranking_df = matching_results_df.merge(
    reviewed_df[['resume_id', 'job_id', 'relevance_label']],
    on=['resume_id', 'job_id']
)

# Create explicitly pre-sorted recommendation ranks
hybrid_sorted_df = hybrid_ranking_df.sort_values(by=["resume_id", "hybrid_score"], ascending=[True, False]).copy()
semantic_sorted_df = hybrid_ranking_df.sort_values(by=["resume_id", "semantic_score"], ascending=[True, False]).copy()

# Save Sorted Rankings
hy_ranking_path = EVALUATION_DIR / "hybrid_ranking_results.csv"
sem_ranking_path = EVALUATION_DIR / "semantic_ranking_results.csv"
hybrid_sorted_df.to_csv(hy_ranking_path, index=False)
semantic_sorted_df.to_csv(sem_ranking_path, index=False)
print(f"[SAVED] Sorted Hybrid Rankings: {hy_ranking_path}")
print(f"[SAVED] Sorted Semantic Rankings: {sem_ranking_path}")

# 5. Executable Validation Assertions
assert len(matching_results_df) == 40, "Expected exactly 40 evaluated pairs."
assert not matching_results_df.duplicated(['resume_id', 'job_id']).any(), "Duplicate pairs detected!"
assert not matching_results_df.isna().any().any(), "Detected unexpected null values!"
assert (matching_results_df['hybrid_score'] >= 0.0).all() and (matching_results_df['hybrid_score'] <= 1.0).all(), "Hybrid scores out of bounds."

# Verify coverage math matches matched_required lengths against strict definitions
for idx, row in matching_results_df.iterrows():
    req_len = len(FINAL_JOB_REQUIREMENTS[row['job_id']]['required'])
    assert req_len > 0, f"Job {row['job_id']} has 0 required skills."
    expected_cov = len(row['matched_required']) / req_len
    assert abs(row['required_coverage'] - expected_cov) < 1e-7, "Required coverage calculation mismatch!"

# Validate sorted order descending for every single resume
for r_id, gp in hybrid_sorted_df.groupby("resume_id"):
    gp_scores = gp["hybrid_score"].to_numpy()
    assert np.all(np.diff(gp_scores) <= 0), f"Sorting validation failed for Hybrid scores on resume {r_id}!"

for r_id, gp in semantic_sorted_df.groupby("resume_id"):
    gp_scores = gp["semantic_score"].to_numpy()
    assert np.all(np.diff(gp_scores) <= 0), f"Sorting validation failed for Semantic scores on resume {r_id}!"

# Validate PostgreSQL alias requirement logic specifically
# R001 has postgresql in resume, and matches J001's general 'sql' requirement without double-counting
r001_j001 = matching_results_df[(matching_results_df['resume_id']=='R001') & (matching_results_df['job_id']=='J001')].iloc[0]
assert 'sql' in r001_j001['matched_required'], "PostgreSQL fallback failed to provide general 'sql' evidence!"
assert r001_j001['matched_required'].count('sql') == 1, "SQL requirement double-counted!"

print("\nAll validation assertions successfully passed!")

# 6. Evaluation metrics computation
def evaluate_metrics(df, score_col):
    results = []
    for r_id, group in df.groupby("resume_id"):
        relevance = group["relevance_label"].to_numpy()
        scores = group[score_col].to_numpy()

        # Sort descending by prediction column
        ranked = group.sort_values(score_col, ascending=False)
        rel_sorted = ranked["relevance_label"].to_numpy()

        # Precision@3
        k = min(3, len(ranked))
        p_at_k = (rel_sorted[:k] >= 1).mean() if k > 0 else 0.0

        # MRR
        rel_indices = np.where(rel_sorted >= 1)[0]
        mrr = 1 / (rel_indices[0] + 1) if len(rel_indices) > 0 else 0.0

        # nDCG
        ndcg = (
            ndcg_score([relevance], [scores])
            if len(group) > 1 and len(np.unique(relevance)) > 1
            else np.nan
        )

        # Spearman Correlation
        corr = (
            spearmanr(relevance, scores).statistic
            if len(group) > 1 and len(np.unique(relevance)) > 1 and len(np.unique(scores)) > 1
            else np.nan
        )

        results.append({
            "resume_id": r_id,
            "precision_at_3": p_at_k,
            "mrr": mrr,
            "ndcg": ndcg,
            "spearman_correlation": corr
        })
    return pd.DataFrame(results)

hybrid_eval = evaluate_metrics(hybrid_ranking_df, 'hybrid_score')
semantic_eval = evaluate_metrics(hybrid_ranking_df, 'semantic_score')

hybrid_eval['model'] = 'Hybrid (Semantic + Coverage)'
semantic_eval['model'] = 'Sentence Transformer (Semantic-Only)'

comparison_summary = pd.concat([hybrid_eval, semantic_eval], ignore_index=True)
comparison_summary_path = EVALUATION_DIR / "model_comparison_metrics_final.csv"
comparison_summary.to_csv(comparison_summary_path, index=False)
print(f"[SAVED] Final Model Comparison Metrics: {comparison_summary_path}")

print("\n=== Mean Performance Metrics Comparison (N=40 reviewed pairs) ===")
display(comparison_summary.groupby('model')[['precision_at_3', 'mrr', 'ndcg', 'spearman_correlation']].mean().round(3))


=== Milestone 3.3: Final Matching Engine Correction ===

[SAVED] Base Skill Matching Results: /content/talentmatch/data/evaluation/skill_matching_results.csv
[SAVED] Sorted Hybrid Rankings: /content/talentmatch/data/evaluation/hybrid_ranking_results.csv
[SAVED] Sorted Semantic Rankings: /content/talentmatch/data/evaluation/semantic_ranking_results.csv

All validation assertions successfully passed!
[SAVED] Final Model Comparison Metrics: /content/talentmatch/data/evaluation/model_comparison_metrics_final.csv

=== Mean Performance Metrics Comparison (N=40 reviewed pairs) ===


,precision_at_3,mrr,ndcg,spearman_correlation
model,,,,
Hybrid (Semantic + Coverage),0.867,1.0,1.0,0.852
Sentence Transformer (Semantic-Only),0.867,1.0,1.0,0.852


In [21]:
# ============================================================
# TALENTMATCH — Milestone 3.3: Audited Matching Engine
# Rebuilds skill matching, evidence, rankings, and evaluation.
# Does NOT modify reviewed_pairs.csv or relevance labels.
# ============================================================

import re
import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

print("=== TALENTMATCH: Audited Matching Engine ===")

# ------------------------------------------------------------
# 1. Paths and input files
# ------------------------------------------------------------

PROJECT_DIR = _resolve_project_dir()
SYNTHETIC_DIR = PROJECT_DIR / "data" / "synthetic"
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

resumes_df = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs_df = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
reviewed_df = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

semantic_path = EVALUATION_DIR / "semantic_similarity_scores.csv"

# Reuse existing semantic scores if available; otherwise load them.
if "semantic_similarity_df" not in globals():
    if not semantic_path.exists():
        raise FileNotFoundError(
            f"Semantic score file not found: {semantic_path}"
        )

    semantic_similarity_df = pd.read_csv(
        semantic_path, index_col=0
    )

# Normalize IDs and labels.
for df in [resumes_df, jobs_df, reviewed_df]:
    for col in ["resume_id", "job_id"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

reviewed_df["relevance_label"] = pd.to_numeric(
    reviewed_df["relevance_label"], errors="raise"
)

assert len(reviewed_df) == 40, (
    f"Expected 40 reviewed pairs; found {len(reviewed_df)}."
)
assert not reviewed_df.duplicated(
    ["resume_id", "job_id"]
).any(), "Duplicate reviewed pairs found."

assert set(reviewed_df["relevance_label"].unique()).issubset(
    {0, 1, 2}
), "Unexpected relevance labels."

# Support a semantic score CSV with either an index or resume_id column.
if "resume_id" in semantic_similarity_df.columns:
    semantic_similarity_df = semantic_similarity_df.set_index("resume_id")

semantic_similarity_df.index = (
    semantic_similarity_df.index.astype(str)
)
semantic_similarity_df.columns = (
    semantic_similarity_df.columns.astype(str)
)

# ------------------------------------------------------------
# 2. Canonical skill patterns
# Each canonical skill has explicit phrase-level aliases.
# Avoid broad patterns that match unrelated concepts.
# ------------------------------------------------------------

SKILL_PATTERNS = {
    "python": [r"\bpython\b"],
    "java": [r"\bjava\b"],
    "oop": [r"\boop\b", r"\bobject[- ]oriented programming\b"],

    "flask": [r"\bflask\b"],
    "fastapi": [r"\bfastapi\b"],
    "rest apis": [
        r"\brest\s+apis?\b",
        r"\bapi\s+endpoints?\b"
    ],

    "sql": [r"\bsql\b"],
    "postgresql": [r"\bpostgresql\b"],
    "git": [r"\bgit\b"],

    "automated testing": [
        r"\bautomated\s+testing\b",
        r"\bunit\s+testing\b"
    ],
    "software testing": [r"\bsoftware\s+testing\b"],
    "debugging": [r"\bdebugging\b"],

    "docker": [r"\bdocker\b"],
    "containerization": [
        r"\bcontainerization\b",
        r"\bcontainerized\b",
        r"\bcontainers?\b"
    ],
    "container orchestration": [
        r"\bcontainer\s+orchestration\b"
    ],

    "pandas": [r"\bpandas\b"],
    "numpy": [r"\bnumpy\b"],
    "excel": [r"\bexcel\b"],
    "power bi": [r"\bpower\s+bi\b"],

    "exploratory data analysis": [
        r"\bexploratory\s+data\s+analysis\b",
        r"\beda\b"
    ],
    "data cleaning": [r"\bdata\s+cleaning\b"],
    "data analysis": [r"\bdata\s+analysis\b"],
    "reporting": [r"\breporting\b", r"\breports?\b"],
    "dashboard creation": [
        r"\bdashboard\s+creation\b",
        r"\bdashboard\s+development\b",
        r"\bdashboards?\b"
    ],

    "scikit-learn": [
        r"\bscikit[- ]learn\b",
        r"\bsklearn\b"
    ],
    "feature engineering": [r"\bfeature\s+engineering\b"],
    "model evaluation": [r"\bmodel\s+evaluation\b"],
    "machine learning fundamentals": [
        r"\bmachine\s+learning\s+fundamentals\b",
        r"\bml\s+fundamentals\b"
    ],
    "machine learning": [
        r"\bmachine\s+learning\b",
        r"\bml\b"
    ],
    "cross-validation": [
        r"\bcross[- ]validation\b"
    ],
    "classification": [r"\bclassification\b"],
    "regression": [r"\bregression\b"],

    "transformers": [r"\btransformers\b"],
    "tokenization": [r"\btokenization\b"],
    "text classification": [r"\btext\s+classification\b"],
    "sentence embeddings": [
        r"\bsentence\s+embeddings?\b"
    ],
    "semantic search": [r"\bsemantic\s+search\b"],
    "hugging face": [r"\bhugging\s+face\b"],
    "pytorch": [r"\bpytorch\b"],
    "nlp": [r"\bnlp\b"],

    "html": [r"\bhtml\b"],
    "css": [r"\bcss\b"],
    "javascript": [r"\bjavascript\b"],
    "react": [r"\breact\b"],
    "responsive web design": [
        r"\bresponsive\s+web\s+design\b",
        r"\bresponsive\s+design\b"
    ],
    "accessibility": [r"\baccessibility\b"],
    "reusable ui components": [
        r"\breusable\s+ui\s+components?\b",
        r"\bui\s+components?\b"
    ],

    "linux": [r"\blinux\b"],
    "ci/cd": [r"\bci\s*/\s*cd\b", r"\bcicd\b"],
    "deployment": [r"\bdeployment\b"],
    "deployment automation": [
        r"\bdeployment\s+automation\b"
    ],
    "aws": [r"\baws\b"],
    "monitoring": [r"\bmonitoring\b"],

    "stakeholder communication": [
        r"\bstakeholder\s+communication\b"
    ],
    "requirements gathering": [
        r"\brequirements\s+gathering\b"
    ],

    "programming fundamentals": [
        r"\bprogramming\s+fundamentals\b"
    ],
    "data structures": [r"\bdata\s+structures\b"],

    "cnns": [r"\bcnns?\b"],
    "image classification": [r"\bimage\s+classification\b"],
    "opencv": [r"\bopencv\b"],
    "transfer learning": [r"\btransfer\s+learning\b"],
    "image augmentation": [r"\bimage\s+augmentation\b"],

    "ai applications": [
        r"\bai\s+applications?\b",
        r"\bartificial intelligence\s+applications?\b"
    ]
}

COMPILED_PATTERNS = {
    skill: [re.compile(p, re.IGNORECASE) for p in patterns]
    for skill, patterns in SKILL_PATTERNS.items()
}


def extract_skill_evidence(text):
    """
    Returns:
      {canonical_skill: [exact matched text, ...]}

    This is phrase-based evidence extraction, not proof of proficiency.
    """
    text = str(text)
    evidence = {}

    for skill, patterns in COMPILED_PATTERNS.items():
        matches = []

        for pattern in patterns:
            for match in pattern.finditer(text):
                matches.append(match.group(0))

        if matches:
            evidence[skill] = sorted(set(matches), key=str.lower)

    return evidence


# ------------------------------------------------------------
# 3. Audited job requirements
# These mappings are manually curated from the supplied
# synthetic job descriptions, not automatically inferred.
# ------------------------------------------------------------

JOB_REQUIREMENTS = {
    "J001": {
        "required": {
            "python", "rest apis", "sql", "git", "automated testing"
        },
        "preferred": {
            "fastapi", "postgresql", "docker", "deployment"
        }
    },

    "J002": {
        "required": {
            "sql", "python", "data cleaning",
            "exploratory data analysis"
        },
        "preferred": {
            "pandas", "excel", "power bi",
            "dashboard creation", "reporting"
        }
    },

    "J003": {
        "required": {
            "python", "scikit-learn", "feature engineering",
            "model evaluation", "machine learning fundamentals"
        },
        "preferred": {
            "pandas", "numpy", "cross-validation",
            "classification", "regression"
        }
    },

    "J004": {
        "required": {
            "python", "nlp", "tokenization",
            "transformers", "text classification"
        },
        "preferred": {
            "hugging face", "pytorch",
            "sentence embeddings", "semantic search"
        }
    },

    "J005": {
        "required": {
            "python", "cnns", "image classification", "pytorch"
        },
        "preferred": {
            "opencv", "image augmentation",
            "transfer learning", "model evaluation"
        }
    },

    "J006": {
        "required": {
            "html", "css", "javascript", "responsive web design"
        },
        "preferred": {
            "react", "accessibility", "git",
            "reusable ui components"
        }
    },

    "J007": {
        "required": {
            "linux", "docker", "ci/cd", "deployment automation"
        },
        "preferred": {
            "aws", "git", "monitoring",
            "container orchestration"
        }
    },

    "J008": {
        "required": {
            "sql", "excel", "reporting", "data analysis"
        },
        "preferred": {
            "power bi", "stakeholder communication",
            "requirements gathering", "dashboard creation"
        }
    },

    "J009": {
        "required": {
            "programming fundamentals", "data structures",
            "debugging", "git", "software testing"
        },
        "preferred": {
            "python", "java", "sql", "oop"
        }
    },

    "J010": {
        "required": {
            "python", "machine learning",
            "model evaluation", "ai applications"
        },
        "preferred": {
            "pytorch", "transformers", "nlp",
            "sentence embeddings", "semantic search"
        }
    }
}

# Ensure every required/preferred skill has an extraction pattern.
all_job_skills = set()

for requirements in JOB_REQUIREMENTS.values():
    all_job_skills.update(requirements["required"])
    all_job_skills.update(requirements["preferred"])

unknown_skills = all_job_skills - set(SKILL_PATTERNS)

assert not unknown_skills, (
    f"Missing extraction patterns: {unknown_skills}"
)

assert set(jobs_df["job_id"]) == set(JOB_REQUIREMENTS), (
    "Job IDs in the source CSV do not match the audited requirement map."
)

# ------------------------------------------------------------
# 4. Extract resume evidence once per resume
# ------------------------------------------------------------

resume_evidence = {}

for _, row in resumes_df.iterrows():
    resume_evidence[row["resume_id"]] = extract_skill_evidence(
        row["resume_text"]
    )

# ------------------------------------------------------------
# 5. PostgreSQL -> SQL evidence fallback
# PostgreSQL remains separately identifiable for preferred
# PostgreSQL requirements. It does not create a second SQL
# requirement or duplicate count.
# ------------------------------------------------------------

def get_effective_evidence(raw_evidence):
    effective = {
        skill: list(matches)
        for skill, matches in raw_evidence.items()
    }

    if "postgresql" in raw_evidence:
        # PostgreSQL is evidence for general SQL.
        # Keep the original PostgreSQL evidence separately.
        effective.setdefault(
            "sql",
            ["PostgreSQL (accepted as SQL evidence)"]
        )

    return effective


# ------------------------------------------------------------
# 6. Rebuild matching results for all reviewed pairs
# ------------------------------------------------------------

results_rows = []

for _, row in reviewed_df.iterrows():

    resume_id = row["resume_id"]
    job_id = row["job_id"]

    raw_evidence = resume_evidence.get(resume_id, {})
    effective_evidence = get_effective_evidence(raw_evidence)

    required = JOB_REQUIREMENTS[job_id]["required"]
    preferred = JOB_REQUIREMENTS[job_id]["preferred"]

    matched_required = required.intersection(effective_evidence)
    missing_required = required - matched_required
    matched_preferred = preferred.intersection(raw_evidence)

    required_coverage = (
        len(matched_required) / len(required)
        if required else 0.0
    )

    semantic_score = float(
        semantic_similarity_df.loc[resume_id, job_id]
    )

    if not np.isfinite(semantic_score):
        raise ValueError(
            f"Invalid semantic score for {resume_id}/{job_id}"
        )

    hybrid_score = (
        0.5 * semantic_score +
        0.5 * required_coverage
    )

    # Keep exact matched phrases for traceability.
    evidence_for_pair = {}

    for skill in sorted(matched_required | matched_preferred):
        evidence_for_pair[skill] = effective_evidence.get(
            skill, []
        )

    explanation = (
        f"Required skills evidenced: {sorted(matched_required)}. "
        f"Preferred skills evidenced: {sorted(matched_preferred)}. "
        f"Required skills not explicitly evidenced: "
        f"{sorted(missing_required)}."
    )

    results_rows.append({
        "resume_id": resume_id,
        "job_id": job_id,
        "matched_required": sorted(matched_required),
        "missing_required": sorted(missing_required),
        "matched_preferred": sorted(matched_preferred),
        "required_coverage": required_coverage,
        "semantic_score": semantic_score,
        "hybrid_score": hybrid_score,
        "skill_evidence": evidence_for_pair,
        "evidence_explanation": explanation
    })


matching_results_df = pd.DataFrame(results_rows)

# ------------------------------------------------------------
# 7. Validation: matching and evidence
# ------------------------------------------------------------

assert len(matching_results_df) == 40
assert not matching_results_df.duplicated(
    ["resume_id", "job_id"]
).any()

assert not matching_results_df.isna().any().any()

assert matching_results_df["semantic_score"].between(
    0, 1
).all()

assert matching_results_df["hybrid_score"].between(
    0, 1
).all()

assert matching_results_df["required_coverage"].between(
    0, 1
).all()

for _, row in matching_results_df.iterrows():

    required = JOB_REQUIREMENTS[row["job_id"]]["required"]

    assert len(required) > 0

    expected_coverage = (
        len(row["matched_required"]) / len(required)
    )

    assert abs(
        row["required_coverage"] - expected_coverage
    ) < 1e-9

    assert set(row["matched_required"]).issubset(required)

    assert set(row["missing_required"]) == (
        required - set(row["matched_required"])
    )

# PostgreSQL alias test using a controlled example.
test_evidence = extract_skill_evidence(
    "Experienced with PostgreSQL database systems."
)

test_effective = get_effective_evidence(test_evidence)

assert "postgresql" in test_evidence
assert "sql" in test_effective
assert len(test_effective["sql"]) == 1

# Actual R001/J001 check.
r001_j001 = matching_results_df[
    (matching_results_df["resume_id"] == "R001") &
    (matching_results_df["job_id"] == "J001")
]

if not r001_j001.empty:
    row = r001_j001.iloc[0]

    assert "sql" in row["matched_required"]
    assert row["matched_required"].count("sql") == 1

    # PostgreSQL is separately available as preferred evidence.
    if "postgresql" in resume_evidence["R001"]:
        assert "postgresql" in row["matched_preferred"]

print("[PASS] Matching, evidence, coverage, and alias checks")

# ------------------------------------------------------------
# 8. Build and save sorted recommendation rankings
# ------------------------------------------------------------

ranking_df = matching_results_df.merge(
    reviewed_df[["resume_id", "job_id", "relevance_label"]],
    on=["resume_id", "job_id"],
    how="inner",
    validate="one_to_one"
)

hybrid_sorted_df = ranking_df.sort_values(
    ["resume_id", "hybrid_score", "job_id"],
    ascending=[True, False, True],
    kind="mergesort"
).copy()

semantic_sorted_df = ranking_df.sort_values(
    ["resume_id", "semantic_score", "job_id"],
    ascending=[True, False, True],
    kind="mergesort"
).copy()

# Convert list/dict columns to JSON strings for clean CSV storage.
def csv_safe_copy(df):
    output = df.copy()

    for col in [
        "matched_required",
        "missing_required",
        "matched_preferred",
        "skill_evidence"
    ]:
        if col in output.columns:
            output[col] = output[col].apply(
                lambda value: json.dumps(value, ensure_ascii=False)
            )

    return output


# Validate score ordering for every resume.
for resume_id, group in hybrid_sorted_df.groupby("resume_id"):
    scores = group["hybrid_score"].to_numpy()
    assert np.all(np.diff(scores) <= 1e-12), (
        f"Hybrid ranking sort failed for {resume_id}"
    )

for resume_id, group in semantic_sorted_df.groupby("resume_id"):
    scores = group["semantic_score"].to_numpy()
    assert np.all(np.diff(scores) <= 1e-12), (
        f"Semantic ranking sort failed for {resume_id}"
    )

# Save base matching results and rankings.
matching_path = EVALUATION_DIR / "skill_matching_results.csv"
hybrid_path = EVALUATION_DIR / "hybrid_ranking_results.csv"
semantic_path_out = EVALUATION_DIR / "semantic_ranking_results.csv"

csv_safe_copy(matching_results_df).to_csv(
    matching_path, index=False
)

csv_safe_copy(hybrid_sorted_df).to_csv(
    hybrid_path, index=False
)

csv_safe_copy(semantic_sorted_df).to_csv(
    semantic_path_out, index=False
)

print("[SAVED]", matching_path)
print("[SAVED]", hybrid_path)
print("[SAVED]", semantic_path_out)
print("[PASS] Both ranking outputs are sorted.")

# ------------------------------------------------------------
# 9. Evaluation
#
# Labels:
# 0 = Low
# 1 = Partial
# 2 = Strong
#
# Report both any-relevance and strong-relevance metrics.
# Only reviewed pairs are evaluated; unreviewed pairs are
# not silently treated as irrelevant.
# ------------------------------------------------------------

def reciprocal_rank(relevance, threshold):
    positions = np.where(relevance >= threshold)[0]

    if len(positions) == 0:
        return 0.0

    return 1.0 / (positions[0] + 1)


def evaluate_model(df, score_column, model_name):

    per_resume = []

    for resume_id, group in df.groupby("resume_id"):

        ranked = group.sort_values(
            [score_column, "job_id"],
            ascending=[False, True],
            kind="mergesort"
        )

        relevance = ranked["relevance_label"].to_numpy()
        scores = ranked[score_column].to_numpy()

        k = min(3, len(ranked))

        precision_any = (
            (relevance[:k] >= 1).mean()
            if k else np.nan
        )

        precision_strong = (
            (relevance[:k] >= 2).mean()
            if k else np.nan
        )

        ndcg = (
            ndcg_score(
                [relevance],
                [scores],
                k=k
            )
            if len(relevance) > 1
            and len(np.unique(relevance)) > 1
            else np.nan
        )

        spearman = (
            spearmanr(relevance, scores).statistic
            if len(relevance) > 1
            and len(np.unique(relevance)) > 1
            and len(np.unique(scores)) > 1
            else np.nan
        )

        per_resume.append({
            "resume_id": resume_id,
            "model": model_name,
            "judged_pairs": len(group),
            "precision_at_3_any": precision_any,
            "precision_at_3_strong": precision_strong,
            "mrr_any": reciprocal_rank(relevance, 1),
            "mrr_strong": reciprocal_rank(relevance, 2),
            "ndcg_at_3": ndcg,
            "spearman_correlation": spearman
        })

    return pd.DataFrame(per_resume)


hybrid_eval = evaluate_model(
    ranking_df,
    "hybrid_score",
    "Hybrid (50% Semantic + 50% Coverage)"
)

semantic_eval = evaluate_model(
    ranking_df,
    "semantic_score",
    "Sentence Transformer (Semantic Only)"
)

comparison_df = pd.concat(
    [hybrid_eval, semantic_eval],
    ignore_index=True
)

metrics_path = (
    EVALUATION_DIR / "model_comparison_metrics_final.csv"
)

comparison_df.to_csv(metrics_path, index=False)

summary = (
    comparison_df
    .groupby("model")[
        [
            "precision_at_3_any",
            "precision_at_3_strong",
            "mrr_any",
            "mrr_strong",
            "ndcg_at_3",
            "spearman_correlation"
        ]
    ]
    .mean()
    .round(3)
)

print("[SAVED]", metrics_path)

print("\n=== Mean Evaluation Metrics ===")
display(summary)

print("\n=== Evaluation Coverage ===")
print("Reviewed pairs:", len(reviewed_df))
print("Possible pairs:", len(resumes_df) * len(jobs_df))
print(
    "Reviewed-pair coverage:",
    f"{len(reviewed_df) / (len(resumes_df) * len(jobs_df)):.1%}"
)

print(
    "\nIMPORTANT: Metrics evaluate only reviewed pairs. "
    "Unreviewed pairs are unknown, not negative examples."
)

print("\n=== FINAL VALIDATION ===")
print("Reviewed pairs:", len(matching_results_df))
print("Unique resumes:", matching_results_df["resume_id"].nunique())
print("Unique jobs:", matching_results_df["job_id"].nunique())
print("Hybrid score range:",
      round(matching_results_df["hybrid_score"].min(), 4),
      "to",
      round(matching_results_df["hybrid_score"].max(), 4))
print("All assertions passed.")

=== TALENTMATCH: Audited Matching Engine ===
[PASS] Matching, evidence, coverage, and alias checks
[SAVED] /content/talentmatch/data/evaluation/skill_matching_results.csv
[SAVED] /content/talentmatch/data/evaluation/hybrid_ranking_results.csv
[SAVED] /content/talentmatch/data/evaluation/semantic_ranking_results.csv
[PASS] Both ranking outputs are sorted.
[SAVED] /content/talentmatch/data/evaluation/model_comparison_metrics_final.csv

=== Mean Evaluation Metrics ===


,precision_at_3_any,precision_at_3_strong,mrr_any,mrr_strong,ndcg_at_3,spearman_correlation
model,,,,,,
Hybrid (50% Semantic + 50% Coverage),0.867,0.367,1.0,1.0,1.0,0.852
Sentence Transformer (Semantic Only),0.867,0.367,1.0,1.0,1.0,0.852



=== Evaluation Coverage ===
Reviewed pairs: 40
Possible pairs: 100
Reviewed-pair coverage: 40.0%

IMPORTANT: Metrics evaluate only reviewed pairs. Unreviewed pairs are unknown, not negative examples.

=== FINAL VALIDATION ===
Reviewed pairs: 40
Unique resumes: 10
Unique jobs: 10
Hybrid score range: 0.0547 to 0.9156
All assertions passed.


In [22]:
# ============================================================
# CELL 16 — TALENTMATCH DATASET AND RANKING AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"
SYNTHETIC_DIR = PROJECT_DIR / "data" / "synthetic"

resumes = pd.read_csv(SYNTHETIC_DIR / "resumes.csv")
jobs = pd.read_csv(SYNTHETIC_DIR / "job_descriptions.csv")
reviewed = pd.read_csv(EVALUATION_DIR / "reviewed_pairs.csv")

matching = pd.read_csv(
    EVALUATION_DIR / "skill_matching_results.csv"
)

hybrid = pd.read_csv(
    EVALUATION_DIR / "hybrid_ranking_results.csv"
)

semantic = pd.read_csv(
    EVALUATION_DIR / "semantic_ranking_results.csv"
)

print("=== CELL 16: DATASET AUDIT ===")

print("\nDataset sizes")
print("Resumes:", len(resumes))
print("Jobs:", len(jobs))
print("Reviewed pairs:", len(reviewed))
print("Possible pairs:", len(resumes) * len(jobs))

print("\nLabel distribution")
print(reviewed["relevance_label"].value_counts().sort_index())

print("\nReviewed pairs per resume")
print(reviewed.groupby("resume_id").size())

print("\nReviewed pairs per job")
print(reviewed.groupby("job_id").size())

print("\n=== TOP HYBRID RECOMMENDATIONS ===")

for resume_id, group in hybrid.groupby("resume_id"):
    top = group.sort_values(
        "hybrid_score", ascending=False
    ).head(3)

    print(f"\n{resume_id}")

    for _, row in top.iterrows():
        print(
            f"  {row['job_id']} | "
            f"score={row['hybrid_score']:.3f} | "
            f"label={row['relevance_label']}"
        )

print("\n=== HYBRID VS SEMANTIC SCORE DIFFERENCES ===")

comparison = hybrid[
    ["resume_id", "job_id", "hybrid_score", "semantic_score"]
].merge(
    semantic[
        ["resume_id", "job_id", "semantic_score"]
    ],
    on=["resume_id", "job_id"],
    suffixes=("_hybrid_file", "_semantic_file")
)

# Check whether the two rankings differ for any resume.
ranking_differences = []

for resume_id in sorted(hybrid["resume_id"].unique()):

    h_order = (
        hybrid[hybrid["resume_id"] == resume_id]
        .sort_values("hybrid_score", ascending=False)["job_id"]
        .tolist()
    )

    s_order = (
        semantic[semantic["resume_id"] == resume_id]
        .sort_values("semantic_score", ascending=False)["job_id"]
        .tolist()
    )

    if h_order != s_order:
        ranking_differences.append(resume_id)

print("Resumes with different ranking order:", ranking_differences)

print("\n=== VALIDATION ===")

assert len(resumes) == 10
assert len(jobs) == 10
assert len(reviewed) == 40
assert len(matching) == 40
assert not reviewed.duplicated(["resume_id", "job_id"]).any()

print("All Cell 16 checks passed.")

=== CELL 16: DATASET AUDIT ===

Dataset sizes
Resumes: 10
Jobs: 10
Reviewed pairs: 40
Possible pairs: 100

Label distribution
relevance_label
0     8
1    21
2    11
Name: count, dtype: int64

Reviewed pairs per resume
resume_id
R001    4
R002    5
R003    5
R004    4
R005    4
R006    4
R007    3
R008    3
R009    4
R010    4
dtype: int64

Reviewed pairs per job
job_id
J001    10
J002    10
J003     5
J004     3
J005     2
J006     1
J007     3
J008     2
J009     2
J010     2
dtype: int64

=== TOP HYBRID RECOMMENDATIONS ===

R001
  J001 | score=0.860 | label=2
  J002 | score=0.517 | label=1
  J009 | score=0.429 | label=1

R002
  J002 | score=0.857 | label=2
  J008 | score=0.787 | label=2
  J001 | score=0.377 | label=1

R003
  J003 | score=0.783 | label=2
  J010 | score=0.628 | label=1
  J002 | score=0.359 | label=1

R004
  J004 | score=0.775 | label=2
  J010 | score=0.320 | label=1
  J002 | score=0.116 | label=0

R005
  J001 | score=0.916 | label=2
  J002 | score=0.487 | label=1
  J0

In [ ]:
# ============================================================
# CELL 17 — SKILL EXTRACTION AND MATCHING UNIT TESTS
# ============================================================

print("=== CELL 17: FOCUSED MATCHING TESTS ===")

# 1. PostgreSQL must provide SQL evidence.
evidence = extract_skill_evidence(
    "Built applications using PostgreSQL."
)
effective = get_effective_evidence(evidence)

assert "postgresql" in evidence
assert "sql" in effective
assert len(effective["sql"]) == 1
print("[PASS] PostgreSQL -> SQL alias")

# 2. Debugging and software testing must remain distinct.
evidence = extract_skill_evidence(
    "Experience in debugging applications and software testing."
)

assert "debugging" in evidence
assert "software testing" in evidence
assert "automated testing" not in evidence
print("[PASS] Debugging and software testing are distinct")

# 3. CNN and image classification.
evidence = extract_skill_evidence(
    "Developed CNN-based image classification models using PyTorch."
)

assert {"cnns", "image classification", "pytorch"}.issubset(evidence)
print("[PASS] Computer vision skill extraction")

# 4. Java and OOP.
evidence = extract_skill_evidence(
    "Developed Java applications using object-oriented programming."
)

assert {"java", "oop"}.issubset(evidence)
print("[PASS] Java and OOP extraction")

# 5. Responsive design and reusable UI components.
evidence = extract_skill_evidence(
    "Created responsive web design with reusable UI components."
)

assert "responsive web design" in evidence
assert "reusable ui components" in evidence
print("[PASS] Frontend skill extraction")

# 6. Avoid generic false positives.
negative_cases = {
    "deployment": ["deployment automation"],
    "container orchestration": ["containerized application"],
    "stakeholder communication": ["stakeholder meeting"],
    "requirements gathering": ["project requirements"]
}

for skill, texts in negative_cases.items():
    for text in texts:
        extracted = extract_skill_evidence(text)
        assert skill not in extracted, (
            f"False positive: {skill} extracted from '{text}'"
        )

print("[PASS] Generic false-positive checks")

# 7. Missing evidence must remain missing, not inferred.
evidence = extract_skill_evidence(
    "Worked on backend projects and database applications."
)

assert "python" not in evidence
assert "sql" not in evidence
print("[PASS] No unsupported skill inference")

# 8. Every job mapping has valid extraction patterns.
for job_id, requirements in JOB_REQUIREMENTS.items():
    for skill in requirements["required"] | requirements["preferred"]:
        assert skill in SKILL_PATTERNS, (
            f"Missing pattern for {job_id}: {skill}"
        )

print("[PASS] All job requirement skills have patterns")

print("\nAll Cell 17 tests passed.")

In [24]:
# ============================================================
# CELL 18 — FINAL EVALUATION AND REPORT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

reviewed = pd.read_csv(
    EVALUATION_DIR / "reviewed_pairs.csv"
)

hybrid = pd.read_csv(
    EVALUATION_DIR / "hybrid_ranking_results.csv"
)

semantic = pd.read_csv(
    EVALUATION_DIR / "semantic_ranking_results.csv"
)

def calculate_final_metrics(df, score_column, model_name):
    rows = []

    for resume_id, group in df.groupby("resume_id"):

        ranked = group.sort_values(
            [score_column, "job_id"],
            ascending=[False, True]
        )

        relevance = ranked["relevance_label"].to_numpy()
        scores = ranked[score_column].to_numpy()

        k = min(3, len(group))

        relevant_any = relevance >= 1
        relevant_strong = relevance >= 2

        positions_any = np.where(relevant_any)[0]
        positions_strong = np.where(relevant_strong)[0]

        mrr_any = (
            1 / (positions_any[0] + 1)
            if len(positions_any) else 0.0
        )

        mrr_strong = (
            1 / (positions_strong[0] + 1)
            if len(positions_strong) else 0.0
        )

        ndcg = (
            ndcg_score([relevance], [scores], k=k)
            if len(group) > 1 and len(np.unique(relevance)) > 1
            else np.nan
        )

        rho = (
            spearmanr(relevance, scores).statistic
            if len(group) > 1
            and len(np.unique(relevance)) > 1
            and len(np.unique(scores)) > 1
            else np.nan
        )

        rows.append({
            "resume_id": resume_id,
            "model": model_name,
            "reviewed_jobs": len(group),
            "precision_at_3_any": relevant_any[:k].mean(),
            "precision_at_3_strong": relevant_strong[:k].mean(),
            "mrr_any": mrr_any,
            "mrr_strong": mrr_strong,
            "ndcg_at_3": ndcg,
            "spearman": rho
        })

    return pd.DataFrame(rows)


hybrid_metrics = calculate_final_metrics(
    hybrid,
    "hybrid_score",
    "Hybrid"
)

semantic_metrics = calculate_final_metrics(
    semantic,
    "semantic_score",
    "Sentence Transformer"
)

all_metrics = pd.concat(
    [hybrid_metrics, semantic_metrics],
    ignore_index=True
)

summary = all_metrics.groupby("model")[
    [
        "precision_at_3_any",
        "precision_at_3_strong",
        "mrr_any",
        "mrr_strong",
        "ndcg_at_3",
        "spearman"
    ]
].mean()

summary = summary.round(3)

print("=== FINAL MODEL COMPARISON ===")
display(summary)

print("\n=== PER-RESUME METRICS ===")
display(all_metrics.round(3))

# Save evaluation outputs.
all_metrics.to_csv(
    EVALUATION_DIR / "final_per_resume_metrics.csv",
    index=False
)

summary.to_csv(
    EVALUATION_DIR / "final_model_summary.csv"
)

# Build Markdown report.
label_counts = reviewed["relevance_label"].value_counts().sort_index()
coverage = len(reviewed) / 100

report = f"""# TALENTMATCH — Evaluation Report

## Dataset

- Synthetic resumes: 10
- Synthetic job descriptions: 10
- Possible resume-job pairs: 100
- Reviewed pairs: {len(reviewed)}
- Reviewed-pair coverage: {coverage:.1%}

## Relevance labels

- Low (0): {label_counts.get(0, 0)}
- Partial (1): {label_counts.get(1, 0)}
- Strong (2): {label_counts.get(2, 0)}

## Model comparison

{summary.to_markdown()}

## Evaluation methodology

- Precision@3 (any): labels 1 and 2 count as relevant.
- Precision@3 (strong): only label 2 counts as relevant.
- MRR (any): first result with label 1 or 2.
- MRR (strong): first result with label 2.
- nDCG@3: graded relevance using labels 0, 1 and 2.
- Spearman: correlation between model scores and reviewed relevance labels.

## Limitations

1. Only {len(reviewed)} of 100 possible pairs have reviewed relevance labels.
2. Unreviewed pairs are unknown, not negative examples.
3. Synthetic resumes and job descriptions do not establish real-world effectiveness.
4. The 50/50 hybrid weighting is a fixed experimental choice, not an optimized weight.
5. Exact phrase extraction can miss valid paraphrases and context.
6. Similarity and skill coverage are decision-support signals, not hiring verdicts.
7. Results should not be generalized to real candidate populations.

## Interpretation

The hybrid model must not be claimed as superior unless it demonstrates
consistent improvement on a larger, independently reviewed evaluation set.
"""

report_path = EVALUATION_DIR / "FINAL_EVALUATION_REPORT.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("\n[SAVED] final_per_resume_metrics.csv")
print("[SAVED] final_model_summary.csv")
print("[SAVED]", report_path)
print("[PASS] Evaluation report generated.")

=== FINAL MODEL COMPARISON ===


,precision_at_3_any,precision_at_3_strong,mrr_any,mrr_strong,ndcg_at_3,spearman
model,,,,,,
Hybrid,0.867,0.367,1.0,1.0,1.0,0.852
Sentence Transformer,0.867,0.367,1.0,1.0,1.0,0.852



=== PER-RESUME METRICS ===


,resume_id,model,reviewed_jobs,precision_at_3_any,precision_at_3_strong,mrr_any,mrr_strong,ndcg_at_3,spearman
0,R001,Hybrid,4,1.000,0.333,1.0,1.0,1.0,0.775
1,R002,Hybrid,5,1.000,0.667,1.0,1.0,1.0,0.949
2,R003,Hybrid,5,1.000,0.333,1.0,1.0,1.0,0.707
3,R004,Hybrid,4,0.667,0.333,1.0,1.0,1.0,0.949
4,R005,Hybrid,4,1.000,0.333,1.0,1.0,1.0,0.775
5,R006,Hybrid,4,0.333,0.333,1.0,1.0,1.0,0.775
6,R007,Hybrid,3,1.000,0.333,1.0,1.0,1.0,0.866
7,R008,Hybrid,3,0.667,0.333,1.0,1.0,1.0,1.000
8,R009,Hybrid,4,1.000,0.333,1.0,1.0,1.0,0.949
9,R010,Hybrid,4,1.000,0.333,1.0,1.0,1.0,0.775



[SAVED] final_per_resume_metrics.csv
[SAVED] final_model_summary.csv
[SAVED] /content/talentmatch/data/evaluation/FINAL_EVALUATION_REPORT.md
[PASS] Evaluation report generated.


In [25]:
# ============================================================
# CELL 19 — PROJECT STRUCTURE AND REPRODUCIBILITY AUDIT
# ============================================================

from pathlib import Path
import importlib.metadata as metadata
import json

PROJECT_DIR = _resolve_project_dir()

print("=== CELL 19: PROJECT AUDIT ===")

# 1. Show current project files.
print("\n--- Existing project files ---")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

# 2. Check core project artifacts.
required_files = {
    "Synthetic resumes":
        PROJECT_DIR / "data/synthetic/resumes.csv",

    "Synthetic job descriptions":
        PROJECT_DIR / "data/synthetic/job_descriptions.csv",

    "Reviewed relevance labels":
        PROJECT_DIR / "data/evaluation/reviewed_pairs.csv",

    "Skill matching results":
        PROJECT_DIR / "data/evaluation/skill_matching_results.csv",

    "Hybrid rankings":
        PROJECT_DIR / "data/evaluation/hybrid_ranking_results.csv",

    "Semantic rankings":
        PROJECT_DIR / "data/evaluation/semantic_ranking_results.csv",

    "Final evaluation report":
        PROJECT_DIR / "data/evaluation/FINAL_EVALUATION_REPORT.md",

    "Per-resume metrics":
        PROJECT_DIR / "data/evaluation/final_per_resume_metrics.csv",

    "Model summary":
        PROJECT_DIR / "data/evaluation/final_model_summary.csv",

    "Annotation rubric":
        PROJECT_DIR / "annotation_rubric.md"
}

print("\n--- Required artifact checks ---")

missing = []

for name, path in required_files.items():
    exists = path.exists()
    print(f"{'[PASS]' if exists else '[MISSING]'} {name}: {path}")

    if not exists:
        missing.append(name)

# 3. Check documentation and engineering files.
optional_files = {
    "README": PROJECT_DIR / "README.md",
    "Requirements": PROJECT_DIR / "requirements.txt",
    "Git ignore": PROJECT_DIR / ".gitignore",
    "Tests directory": PROJECT_DIR / "tests",
    "Application directory": PROJECT_DIR / "app",
    "Notebook": PROJECT_DIR / "TALENTMATCH.ipynb"
}

print("\n--- Engineering readiness ---")

for name, path in optional_files.items():
    print(
        f"{'[FOUND]' if path.exists() else '[NOT FOUND]'} "
        f"{name}: {path}"
    )

# 4. Check lightweight dependency availability.
packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "scipy",
    "sentence-transformers",
    "torch",
    "transformers",
    "streamlit",
    "pytest"
]

print("\n--- Dependency availability ---")

for package in packages:
    try:
        version = metadata.version(package)
        print(f"[INSTALLED] {package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"[NOT INSTALLED] {package}")

# 5. Check for accidental secret-like files.
secret_candidates = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.name.lower() in {
        ".env", "credentials.json", "service-account.json"
    }:
        secret_candidates.append(str(path.relative_to(PROJECT_DIR)))

print("\n--- Secret-file filename scan ---")

if secret_candidates:
    print("[WARNING] Review these files manually:", secret_candidates)
else:
    print("[PASS] No common secret filenames detected.")

# This filename scan is not a complete secret scan.

# 6. Save audit report.
audit = {
    "project": "TALENTMATCH",
    "required_artifacts": {
        name: path.exists()
        for name, path in required_files.items()
    },
    "missing_artifacts": missing,
    "engineering_files": {
        name: path.exists()
        for name, path in optional_files.items()
    },
    "secret_filename_candidates": secret_candidates
}

audit_path = PROJECT_DIR / "PROJECT_AUDIT.json"

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("\n[SAVED]", audit_path)
print("\n=== AUDIT SUMMARY ===")
print("Missing required artifacts:", len(missing))

if missing:
    print("Missing:", missing)
else:
    print("All checked core artifacts exist.")

print(
    "\nNote: File existence does not prove tests, deployment, "
    "GitHub push, or reproducibility."
)

=== CELL 19: PROJECT AUDIT ===

--- Existing project files ---
annotation_rubric.md
data/evaluation/FINAL_EVALUATION_REPORT.md
data/evaluation/final_model_summary.csv
data/evaluation/final_per_resume_metrics.csv
data/evaluation/hybrid_ranking_results.csv
data/evaluation/model_comparison_metrics.csv
data/evaluation/model_comparison_metrics_40_pairs.csv
data/evaluation/model_comparison_metrics_corrected.csv
data/evaluation/model_comparison_metrics_final.csv
data/evaluation/model_comparison_pair_scores.csv
data/evaluation/model_comparison_scores_40_pairs.csv
data/evaluation/remaining_pairs_for_review.csv
data/evaluation/review_batch_20.csv
data/evaluation/reviewed_pairs.csv
data/evaluation/semantic_ranking_results.csv
data/evaluation/semantic_similarity_scores.csv
data/evaluation/skill_matching_results.csv
data/evaluation/tfidf_metrics_by_resume.csv
data/evaluation/tfidf_pair_scores.csv
data/synthetic/job_descriptions.csv
data/synthetic/resumes.csv

--- Required artifact checks ---
[PASS]

In [26]:
print("=== CELL 17B: DEPLOYMENT EXTRACTION TESTS ===")

# Positive case: explicit deployment evidence must be detected.
positive_text = "Built deployment automation for Python applications."
positive_result = extract_skill_evidence(positive_text)

assert "deployment" in positive_result, (
    "Expected deployment to be extracted from explicit deployment evidence."
)
print("[PASS] Explicit deployment evidence detected")

# Negative case: generic project coordination is not deployment evidence.
negative_text = "Worked with the team on project planning and coordination."
negative_result = extract_skill_evidence(negative_text)

assert "deployment" not in negative_result, (
    "False positive: deployment extracted without deployment evidence."
)
print("[PASS] Generic project coordination does not imply deployment")

print("\n[PASS] Deployment extraction tests completed.")

=== CELL 17B: DEPLOYMENT EXTRACTION TESTS ===
[PASS] Explicit deployment evidence detected
[PASS] Generic project coordination does not imply deployment

[PASS] Deployment extraction tests completed.


In [27]:
from pathlib import Path

ROOT = PROJECT_DIR

# Create directories
for folder in [
    "tests",
    "app",
    "reports",
    "models"
]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

# requirements.txt
requirements = """pandas
numpy
scikit-learn
scipy
sentence-transformers
torch
streamlit
pytest
"""

(ROOT / "requirements.txt").write_text(requirements)

# .gitignore
gitignore = """# Python
__pycache__/
*.py[cod]
.pytest_cache/
.ipynb_checkpoints/

# Environments and secrets
.env
.env.*
!.env.example
credentials.json
service-account.json

# Model/cache files
.cache/
models/
*.pt
*.pth
*.bin

# Temporary files
*.log
.DS_Store
"""

(ROOT / ".gitignore").write_text(gitignore)

# Project metadata
metadata = """# TALENTMATCH

Transformer-based resume and job-description matching with
evidence-backed skill extraction and explainable skill-gap analysis.

Project status: Evaluation and packaging in progress.
Dataset: Synthetic resumes and job descriptions.
"""

(ROOT / "PROJECT_OVERVIEW.md").write_text(metadata)

print("=== CELL 20: PROJECT STRUCTURE ===")

for path in [
    ROOT / "tests",
    ROOT / "app",
    ROOT / "reports",
    ROOT / "models",
    ROOT / "requirements.txt",
    ROOT / ".gitignore",
    ROOT / "PROJECT_OVERVIEW.md"
]:
    print("[PASS]" if path.exists() else "[FAIL]", path)

print("\n[PASS] Project structure created.")

=== CELL 20: PROJECT STRUCTURE ===
[PASS] /content/talentmatch/tests
[PASS] /content/talentmatch/app
[PASS] /content/talentmatch/reports
[PASS] /content/talentmatch/models
[PASS] /content/talentmatch/requirements.txt
[PASS] /content/talentmatch/.gitignore
[PASS] /content/talentmatch/PROJECT_OVERVIEW.md

[PASS] Project structure created.


In [28]:
from pathlib import Path
import pandas as pd

ROOT = PROJECT_DIR
EVAL = ROOT / "data" / "evaluation"
SYNTHETIC = ROOT / "data" / "synthetic"

FILES = {
    "resumes": SYNTHETIC / "resumes.csv",
    "jobs": SYNTHETIC / "job_descriptions.csv",
    "skill_results": EVAL / "skill_matching_results.csv",
    "hybrid_rankings": EVAL / "hybrid_ranking_results.csv",
    "semantic_rankings": EVAL / "semantic_ranking_results.csv",
    "reviewed_pairs": EVAL / "reviewed_pairs.csv",
}

print("=== CELL 21: APP INPUT INSPECTION ===")

dataframes = {}

for name, path in FILES.items():
    print(f"\n--- {name.upper()} ---")

    if not path.exists():
        print("[MISSING]", path)
        continue

    df = pd.read_csv(path)
    dataframes[name] = df

    print("Path:", path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nSample:")
    display(df.head(2))

print("\n=== MODEL / NOTEBOOK ARTIFACTS ===")

for path in [
    ROOT / "TALENTMATCH.ipynb",
    ROOT / "models",
    ROOT / "data" / "evaluation" / "semantic_similarity_scores.csv",
    ROOT / "data" / "evaluation" / "FINAL_EVALUATION_REPORT.md",
]:
    if path.is_dir():
        contents = list(path.iterdir())
        print(f"[DIRECTORY] {path}")
        print("Contents:", [p.name for p in contents] or "Empty")
    else:
        print("[FOUND]" if path.exists() else "[NOT FOUND]", path)

print("\n=== APP READINESS CHECK ===")

required = [
    "resumes",
    "jobs",
    "skill_results",
    "hybrid_rankings",
    "semantic_rankings",
]

for name in required:
    print(
        "[PASS]" if name in dataframes else "[FAIL]",
        name
    )

print("\nInspection complete. No source files were modified.")

=== CELL 21: APP INPUT INSPECTION ===

--- RESUMES ---
Path: /content/talentmatch/data/synthetic/resumes.csv
Shape: (10, 3)
Columns: ['resume_id', 'profile', 'resume_text']

Sample:


,resume_id,profile,resume_text
0,R001,Python Developer,\n Python developer with experience in ...
1,R002,Data Analyst,"\n Data analyst skilled in SQL, Python,..."



--- JOBS ---
Path: /content/talentmatch/data/synthetic/job_descriptions.csv
Shape: (10, 3)
Columns: ['job_id', 'job_title', 'job_description']

Sample:


,job_id,job_title,job_description
0,J001,Python Backend Developer,"\n Required: Python, REST APIs, SQL, Gi..."
1,J002,Data Analyst,"\n Required: SQL, Python, data cleaning..."



--- SKILL_RESULTS ---
Path: /content/talentmatch/data/evaluation/skill_matching_results.csv
Shape: (40, 10)
Columns: ['resume_id', 'job_id', 'matched_required', 'missing_required', 'matched_preferred', 'required_coverage', 'semantic_score', 'hybrid_score', 'skill_evidence', 'evidence_explanation']

Sample:


,resume_id,job_id,matched_required,missing_required,matched_preferred,required_coverage,semantic_score,hybrid_score,skill_evidence,evidence_explanation
0,R001,J001,"[""automated testing"", ""git"", ""python"", ""rest a...",[],"[""docker"", ""postgresql""]",1.0,0.720214,0.860107,"{""automated testing"": [""unit testing""], ""docke...",Required skills evidenced: ['automated testing...
1,R002,J002,"[""data cleaning"", ""exploratory data analysis"",...",[],"[""dashboard creation"", ""excel"", ""pandas"", ""pow...",1.0,0.713044,0.856522,"{""dashboard creation"": [""dashboard"", ""dashboar...","Required skills evidenced: ['data cleaning', '..."


... (3 outputs left out)

,resume_id,job_id,matched_required,missing_required,matched_preferred,required_coverage,semantic_score,hybrid_score,skill_evidence,evidence_explanation,relevance_label
0,R001,J001,"[""automated testing"", ""git"", ""python"", ""rest a...",[],"[""docker"", ""postgresql""]",1.0,0.720214,0.860107,"{""automated testing"": [""unit testing""], ""docke...",Required skills evidenced: ['automated testing...,2
1,R001,J009,"[""git""]","[""data structures"", ""debugging"", ""programming ...","[""python"", ""sql""]",0.2,0.658092,0.429046,"{""git"": [""Git""], ""python"": [""Python""], ""sql"": ...",Required skills evidenced: ['git']. Preferred ...,1



--- REVIEWED_PAIRS ---
Path: /content/talentmatch/data/evaluation/reviewed_pairs.csv
Shape: (40, 4)
Columns: ['resume_id', 'job_id', 'relevance_label', 'label_reason']

Sample:


,resume_id,job_id,relevance_label,label_reason
0,R001,J001,2,"Python, REST APIs, SQL, Git, and testing align..."
1,R002,J002,2,"SQL, Python, data cleaning, analysis, and repo..."



=== MODEL / NOTEBOOK ARTIFACTS ===
[NOT FOUND] /content/talentmatch/TALENTMATCH.ipynb
[DIRECTORY] /content/talentmatch/models
Contents: Empty
[FOUND] /content/talentmatch/data/evaluation/semantic_similarity_scores.csv
[FOUND] /content/talentmatch/data/evaluation/FINAL_EVALUATION_REPORT.md

=== APP READINESS CHECK ===
[PASS] resumes
[PASS] jobs
[PASS] skill_results
[PASS] hybrid_rankings
[PASS] semantic_rankings

Inspection complete. No source files were modified.


In [29]:
from pathlib import Path

ROOT = PROJECT_DIR
APP_DIR = ROOT / "app"
APP_DIR.mkdir(exist_ok=True)

app_code = r'''
from pathlib import Path
import ast
import json

import pandas as pd
import streamlit as st

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"

RESUME_FILE = DATA / "synthetic" / "resumes.csv"
JOBS_FILE = DATA / "synthetic" / "job_descriptions.csv"
HYBRID_FILE = DATA / "evaluation" / "hybrid_ranking_results.csv"
SEMANTIC_FILE = DATA / "evaluation" / "semantic_ranking_results.csv"

st.set_page_config(
    page_title="TALENTMATCH",
    page_icon="🎯",
    layout="wide"
)

@st.cache_data
def load_data():
    resumes = pd.read_csv(RESUME_FILE)
    jobs = pd.read_csv(JOBS_FILE)
    hybrid = pd.read_csv(HYBRID_FILE)
    semantic = pd.read_csv(SEMANTIC_FILE)
    return resumes, jobs, hybrid, semantic

def parse_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, TypeError):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []

def parse_evidence(value):
    if pd.isna(value) or not str(value).strip():
        return {}
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, TypeError):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, dict) else {}
        except (ValueError, SyntaxError):
            return {}

st.title("🎯 TALENTMATCH")
st.subheader("Evidence-backed resume and job matching")

st.write(
    "Explore estimated relevance between synthetic resumes and job descriptions. "
    "Inspect skill coverage, missing requirements, and supporting text evidence."
)

st.warning(
    "Research/demo tool only. Scores are not hiring probabilities or suitability "
    "verdicts. Missing evidence does not prove a person lacks a skill. "
    "The dataset is synthetic and evaluation coverage is limited."
)

try:
    resumes, jobs, hybrid, semantic = load_data()
except Exception as exc:
    st.error(f"Could not load project data: {exc}")
    st.stop()

resume_options = {
    f"{row.resume_id} — {row.profile}": row.resume_id
    for row in resumes.itertuples()
}

selected_label = st.sidebar.selectbox(
    "Select synthetic resume",
    list(resume_options.keys())
)
resume_id = resume_options[selected_label]

ranking_method = st.sidebar.radio(
    "Ranking method",
    ["Hybrid", "Sentence Transformer"]
)

ranking_df = hybrid if ranking_method == "Hybrid" else semantic
score_column = "hybrid_score" if ranking_method == "Hybrid" else "semantic_score"

resume_row = resumes[resumes["resume_id"] == resume_id].iloc[0]

st.markdown("### Selected resume")
st.caption(f"Resume ID: {resume_id} | Profile: {resume_row['profile']}")
with st.expander("View synthetic resume text"):
    st.write(resume_row["resume_text"])

results = ranking_df[ranking_df["resume_id"] == resume_id].copy()
results = results.sort_values(score_column, ascending=False)

if results.empty:
    st.info("No saved ranking results are available for this resume.")
    st.stop()

st.markdown("### Ranked job matches")
st.caption(
    f"Ranking method: {ranking_method} | "
    f"Available results: {len(results)}"
)

for rank, (_, result) in enumerate(results.iterrows(), start=1):
    job_rows = jobs[jobs["job_id"] == result["job_id"]]
    if job_rows.empty:
        continue

    job = job_rows.iloc[0]
    score = float(result[score_column])
    coverage = float(result["required_coverage"])

    with st.container(border=True):
        st.markdown(f"#### {rank}. {job['job_title']}")
        st.caption(f"Job ID: {result['job_id']}")

        col1, col2 = st.columns(2)
        col1.metric("Estimated relevance score", f"{score:.3f}")
        col2.metric("Required skill coverage", f"{coverage:.0%}")

        with st.expander("Job description"):
            st.write(job["job_description"])

        matched = parse_list(result["matched_required"])
        missing = parse_list(result["missing_required"])
        preferred = parse_list(result["matched_preferred"])
        evidence = parse_evidence(result["skill_evidence"])

        left, right = st.columns(2)

        with left:
            st.markdown("**Required skills with evidence**")
            if matched:
                for skill in matched:
                    st.write(f"✓ {skill}")
            else:
                st.write("No required skills matched in the saved analysis.")

            st.markdown("**Matched preferred skills**")
            if preferred:
                for skill in preferred:
                    st.write(f"• {skill}")
            else:
                st.write("No preferred skills matched.")

        with right:
            st.markdown("**Required skills without detected evidence**")
            if missing:
                for skill in missing:
                    st.write(f"• {skill}")
            else:
                st.write("No missing required skills detected.")

        with st.expander("Inspect supporting evidence"):
            if evidence:
                for skill, phrases in evidence.items():
                    st.markdown(f"**{skill}**")
                    if isinstance(phrases, list):
                        for phrase in phrases:
                            st.write(f"- “{phrase}”")
                    else:
                        st.write(str(phrases))
            else:
                st.write("No phrase-level evidence stored for this result.")

        explanation = result.get("evidence_explanation", "")
        if pd.notna(explanation) and str(explanation).strip():
            st.caption(str(explanation))

st.divider()
st.caption(
    "TALENTMATCH | Synthetic demonstration dataset | "
    "Estimated relevance, not a hiring decision."
)
'''

app_path = APP_DIR / "streamlit_app.py"
app_path.write_text(app_code, encoding="utf-8")

print("=== CELL 22: STREAMLIT APP FILE ===")
print("Created:", app_path)
print("File size:", app_path.stat().st_size, "bytes")
print("Exists:", app_path.exists())
print("\nNext: run the syntax check in Cell 23.")

=== CELL 22: STREAMLIT APP FILE ===
Created: /content/talentmatch/app/streamlit_app.py
File size: 5911 bytes
Exists: True

Next: run the syntax check in Cell 23.


In [30]:
import ast
from pathlib import Path

app_path = PROJECT_DIR / "app" / "streamlit_app.py"
source = app_path.read_text(encoding="utf-8")

try:
    ast.parse(source)
    print("[PASS] Streamlit app Python syntax is valid.")
except SyntaxError as exc:
    print("[FAIL] Syntax error:", exc)
    raise

required_terms = [
    "hybrid_ranking_results.csv",
    "semantic_ranking_results.csv",
    "matched_required",
    "missing_required",
    "skill_evidence",
    "not hiring probabilities",
]

for term in required_terms:
    assert term in source, f"Missing expected app feature: {term}"
    print("[PASS]", term)

print("\n[PASS] App source checks completed.")

[PASS] Streamlit app Python syntax is valid.
[PASS] hybrid_ranking_results.csv
[PASS] semantic_ranking_results.csv
[PASS] matched_required
[PASS] missing_required
[PASS] skill_evidence
[PASS] not hiring probabilities

[PASS] App source checks completed.


In [31]:
%pip -q install streamlit

In [32]:
import streamlit
print("Streamlit version:", streamlit.__version__)

Streamlit version: 1.67.0


In [33]:
import subprocess
import sys
import time
import requests
from pathlib import Path
from google.colab import output

ROOT = PROJECT_DIR
APP = ROOT / "app" / "streamlit_app.py"
LOG = ROOT / "streamlit.log"

assert APP.exists(), f"App not found: {APP}"

# Stop an earlier Streamlit process from this notebook, if present.
if "streamlit_process" in globals():
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=10)

log_file = open(LOG, "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP),
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit health endpoint.
healthy = False

for attempt in range(30):
    if streamlit_process.poll() is not None:
        break

    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if response.status_code == 200:
            healthy = True
            break
    except requests.RequestException:
        pass

    time.sleep(1)

log_file.flush()

if healthy:
    print("[PASS] Streamlit server is healthy.")
    print("Open the embedded app below.")
    output.serve_kernel_port_as_iframe(8501, height=900)
else:
    print("[FAIL] Streamlit did not become healthy.")
    print("Process status:", streamlit_process.poll())
    print("\n--- Streamlit log ---")
    print(LOG.read_text(errors="replace")[-10000:])

[PASS] Streamlit server is healthy.
Open the embedded app below.


<IPython.core.display.Javascript object>

In [34]:
import requests
import subprocess
from pathlib import Path

ROOT = PROJECT_DIR
LOG = ROOT / "streamlit.log"
BASE = "http://127.0.0.1:8501"

print("=== CELL 26: STREAMLIT DIAGNOSTICS ===")

# 1. Process status
if "streamlit_process" in globals():
    print("Process running:", streamlit_process.poll() is None)
    print("Process exit code:", streamlit_process.poll())
else:
    print("[FAIL] Streamlit process variable not found.")

# 2. HTTP endpoints
for endpoint in ["/_stcore/health", "/"]:
    try:
        response = requests.get(BASE + endpoint, timeout=10)
        print(
            f"\n{endpoint}: HTTP {response.status_code}"
        )
        print("Content type:", response.headers.get("content-type"))
        print("Response preview:", response.text[:500])
    except Exception as exc:
        print(f"\n[ERROR] {endpoint}: {exc}")

# 3. Server logs
print("\n=== STREAMLIT LOG (LAST 12000 CHARACTERS) ===")

if LOG.exists():
    print(LOG.read_text(errors="replace")[-12000:])
else:
    print("[FAIL] Log file not found.")

# 4. Confirm data files are readable
import pandas as pd

paths = [
    ROOT / "data/synthetic/resumes.csv",
    ROOT / "data/synthetic/job_descriptions.csv",
    ROOT / "data/evaluation/hybrid_ranking_results.csv",
    ROOT / "data/evaluation/semantic_ranking_results.csv",
]

print("\n=== CSV READ TEST ===")

for path in paths:
    try:
        df = pd.read_csv(path)
        print(f"[PASS] {path.name}: {df.shape}")
    except Exception as exc:
        print(f"[FAIL] {path.name}: {exc}")

=== CELL 26: STREAMLIT DIAGNOSTICS ===
Process running: True
Process exit code: None

/_stcore/health: HTTP 200
Content type: text/plain; charset=utf-8
Response preview: ok

/: HTTP 200
Content type: text/html; charset=utf-8
Response preview: <!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or im

=== STREAMLIT LOG (LAST 12000 CHARACTERS) ===
host=gpu-t4-s-usw4a1-1k9efmfjhp4b6.us-west4-a.c.codatalab-user-runtimes.internal:8007
2026-11-09 11:42:39.959 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-

In [35]:
import subprocess
import sys
import time
import requests
from pathlib import Path
from google.colab import output

ROOT = PROJECT_DIR
APP = ROOT / "app" / "streamlit_app.py"
LOG = ROOT / "streamlit.log"

# Stop previous Streamlit process
if "streamlit_process" in globals():
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        try:
            streamlit_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            streamlit_process.kill()
            streamlit_process.wait()

# Start Streamlit with settings for the Colab proxy
log_file = open(LOG, "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP),
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for server health
healthy = False

for _ in range(30):
    if streamlit_process.poll() is not None:
        break

    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=3
        )
        if response.status_code == 200:
            healthy = True
            break
    except requests.RequestException:
        pass

    time.sleep(1)

log_file.flush()

print("Process running:", streamlit_process.poll() is None)
print("Health check:", "PASS" if healthy else "FAIL")

if healthy:
    print("Opening TALENTMATCH...")
    output.serve_kernel_port_as_iframe(8501, height=1000)
else:
    print(LOG.read_text(errors="replace")[-10000:])

Process running: True
Health check: PASS
Opening TALENTMATCH...


<IPython.core.display.Javascript object>

In [36]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

ROOT = PROJECT_DIR
EVAL = ROOT / "data" / "evaluation"

def parse_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        result = json.loads(value)
    except (ValueError, TypeError):
        result = []
    return result if isinstance(result, list) else []

files = {
    "Hybrid": EVAL / "hybrid_ranking_results.csv",
    "Sentence Transformer": EVAL / "semantic_ranking_results.csv",
}

print("=== CELL 28: SKILL COVERAGE VALIDATION ===")

all_passed = True

for model, path in files.items():
    df = pd.read_csv(path)
    errors = []

    for _, row in df.iterrows():
        matched = parse_list(row["matched_required"])
        missing = parse_list(row["missing_required"])

        total = len(matched) + len(missing)
        expected = len(matched) / total if total else 0.0
        actual = float(row["required_coverage"])

        if not np.isclose(expected, actual, atol=1e-6):
            errors.append({
                "resume_id": row["resume_id"],
                "job_id": row["job_id"],
                "matched": len(matched),
                "missing": len(missing),
                "expected": expected,
                "actual": actual,
            })

    if errors:
        all_passed = False
        print(f"\n[FAIL] {model}: {len(errors)} coverage mismatches")
        display(pd.DataFrame(errors))
    else:
        print(f"[PASS] {model}: all {len(df)} rows have consistent coverage")

print("\nFINAL:", "PASS" if all_passed else "FAIL")

=== CELL 28: SKILL COVERAGE VALIDATION ===
[PASS] Hybrid: all 40 rows have consistent coverage
[PASS] Sentence Transformer: all 40 rows have consistent coverage

FINAL: PASS


In [43]:
from pathlib import Path

ROOT = PROJECT_DIR
DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

readme = r"""
# TALENTMATCH

**Transformer-based resume and job-description matching with evidence-backed skill analysis.**

TALENTMATCH compares synthetic resumes with job descriptions, ranks available matches, and explains detected required and preferred skills using text evidence.

> **Important:** This is a research and portfolio demonstration, not a hiring decision system. Scores are estimated relevance scores, not probabilities of suitability. Missing evidence does not prove a candidate lacks a skill.

## 1. Problem

Manually comparing resumes with job descriptions can be time-consuming. TALENTMATCH demonstrates how semantic text similarity and explicit skill evidence can support an initial comparison.

## 2. Features

- Sentence Transformer semantic matching using `all-MiniLM-L6-v2`.
- TF-IDF baseline used during model comparison.
- Rule-based skill extraction with phrase-level evidence.
- Required and preferred skill matching.
- Required-skill coverage and missing-evidence reporting.
- Hybrid ranking combining semantic similarity and required-skill coverage.
- Comparison of hybrid and Sentence Transformer rankings.
- Streamlit interface for exploring saved ranking results.

## 3. System workflow

1. Load synthetic resume and job-description text.
2. Generate semantic representations using Sentence Transformers.
3. Calculate semantic similarity scores.
4. Extract skill evidence using explicit skill patterns and aliases.
5. Calculate required-skill coverage.
6. Combine semantic similarity and coverage for hybrid ranking.
7. Display rankings and supporting evidence.

See [`docs/architecture.md`](docs/architecture.md).

## 4. Scoring

The hybrid score is:

`Hybrid Score = 0.5 × Semantic Similarity + 0.5 × Required Skill Coverage`

Required skill coverage:

`Coverage = Matched Required Skills / Total Required Skills`

The weights are experimental choices, not learned or validated optimal weights.

## 5. Dataset

The current demonstration uses:
- 10 synthetic resumes.
- 10 synthetic job descriptions.
- 100 possible resume/job pairs.
- 40 manually reviewed relevance labels.

The data is synthetic and was created for controlled project development. It is not a representative sample of real applicants or job postings. No real-world hiring effectiveness is established.

Relevance labels:
- `0`: Low
- `1`: Partial
- `2`: Strong

The 60 unreviewed pairs are unknown, not negative examples.

## 6. Evaluation

The current evaluation uses the 40 reviewed pairs.

| Metric | Hybrid | Sentence Transformer |
|---|---:|---:|
| Precision@3 (any relevant) | 0.867 | 0.867 |
| Precision@3 (strong only) | 0.367 | 0.367 |
| MRR (any relevant) | 1.000 | 1.000 |
| MRR (strong only) | 1.000 | 1.000 |
| nDCG@3 | 1.000 | 1.000 |
| Spearman correlation | 0.852 | 0.852 |

These are preliminary results on a small, partially judged synthetic dataset. They do not establish generalization to real resumes or job markets.

The hybrid method did not demonstrate an improvement over Sentence Transformer alone in this evaluation.

Detailed results:
- [`FINAL_EVALUATION_REPORT.md`](data/evaluation/FINAL_EVALUATION_REPORT.md)
- [`final_per_resume_metrics.csv`](data/evaluation/final_per_resume_metrics.csv)
- [`final_model_summary.csv`](data/evaluation/final_model_summary.csv)

## 7. Run locally

Python 3.10 or later is recommended.

```bash
git clone <YOUR_REPOSITORY_URL>
cd TALENTMATCH

python -m venv .venv
```
"""

In [44]:
# Write README.md
readme_path = ROOT / "README.md"
readme_path.write_text(readme.strip() + "\n", encoding="utf-8")

print("[PASS] README created:", readme_path)
print("File exists:", readme_path.exists())
print("File size:", readme_path.stat().st_size, "bytes")

[PASS] README created: /content/talentmatch/README.md
File exists: True
File size: 3426 bytes


In [47]:
from pathlib import Path

ROOT = PROJECT_DIR
DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

architecture = r"""
# TALENTMATCH — System Architecture

## 1. Architecture overview

```mermaid
flowchart TD
    A[Synthetic Resumes] --> C[Text Preparation]
    B[Synthetic Job Descriptions] --> C
    C --> D[Sentence Transformer: all-MiniLM-L6-v2]
    D --> E[Semantic Similarity]
    C --> F[Rule-Based Skill Extraction]
    F --> G[Required and Preferred Skill Evidence]
    G --> H[Required Skill Coverage]
    E --> I[Hybrid Scoring]
    H --> I
    E --> J[Semantic-Only Ranking]
    I --> K[Hybrid Ranking]
    J --> L[Saved Evaluation Results]
    K --> L
    G --> L
    L --> M[Streamlit Demo]
    2. Main components
Component	Responsibility
Synthetic data	Supplies controlled resume and job-description examples
Sentence Transformer	Generates 384-dimensional text embeddings
Semantic similarity	Measures semantic closeness between resume and job text
Skill extraction	Detects configured skills and aliases in resume text
Evidence mapping	Stores matched phrases supporting detected skills
Coverage calculation	Calculates matched required skills divided by total required skills
Hybrid ranking	Combines semantic similarity and required-skill coverage
Evaluation	Compares rankings against manually reviewed relevance labels
Streamlit app	Displays saved rankings, skill coverage, and evidence

3. Scoring
Hybrid score:

0.5 × Semantic Similarity + 0.5 × Required Skill Coverage

Required-skill coverage:

Matched Required Skills / Total Required Skills

These weights are experimental choices, not learned optimal parameters.
4. Data flow
1. Load resume and job-description datasets.
2. Encode text with the Sentence Transformer.
3. Calculate semantic similarity scores.
4. Detect required and preferred skills using configured patterns.
5. Retain supporting text evidence for detected skills.
6. Calculate required-skill coverage.
7. Generate semantic-only and hybrid rankings.
8. Compare rankings against reviewed relevance labels.
9. Save evaluation artifacts for the Streamlit application.
5. Technology
- Python
- pandas and NumPy
- scikit-learn
- Sentence Transformers
- PyTorch
- Streamlit
- pytest
6. Limitations and responsible use
- Current demonstration data is synthetic.
- Only 40 of 100 possible resume/job pairs have reviewed labels.
- Unreviewed pairs are unknown, not negative.
- Evaluation metrics are preliminary and do not establish real-world effectiveness.
- Skill extraction is pattern-based and may miss paraphrases or context.
- Missing detected evidence does not prove a person lacks a skill.
- Scores are estimated relevance measures, not hiring probabilities or suitability verdicts.
- Human review is required for any consequential decision.
  """
path = DOCS / "architecture.md"
path.write_text(architecture.strip() + "\n", encoding="utf-8")

print("[PASS] Architecture document created:", path)
print("File size:", path.stat().st_size, "bytes")
print("[PASS] Mermaid diagram included:", "```mermaid" in architecture)

[PASS] Architecture document created: /content/talentmatch/docs/architecture.md
File size: 2664 bytes
[PASS] Mermaid diagram included: True


In [49]:

### Cell 31 — Create project dependencies


from pathlib import Path

ROOT = PROJECT_DIR

requirements = """# TALENTMATCH dependencies
# Core data processing
numpy
pandas

# Machine learning and evaluation
scikit-learn
scipy

# Transformer embeddings and model development
sentence-transformers
torch

# Interactive demonstration
streamlit

# Testing
pytest
"""

path = ROOT / "requirements.txt"
path.write_text(requirements, encoding="utf-8")

print("[PASS] requirements.txt created")
print(path.read_text(encoding="utf-8"))

[PASS] requirements.txt created
# TALENTMATCH dependencies
# Core data processing
numpy
pandas

# Machine learning and evaluation
scikit-learn
scipy

# Transformer embeddings and model development
sentence-transformers
torch

# Interactive demonstration
streamlit

# Testing
pytest



In [50]:
from pathlib import Path

ROOT = PROJECT_DIR
TESTS = ROOT / "tests"
TESTS.mkdir(exist_ok=True)

test_code = r'''
from pathlib import Path
import json
import pandas as pd

ROOT = PROJECT_DIR
DATA = ROOT / "data"

def test_required_project_files_exist():
    required = [
        DATA / "synthetic/resumes.csv",
        DATA / "synthetic/job_descriptions.csv",
        DATA / "evaluation/reviewed_pairs.csv",
        DATA / "evaluation/skill_matching_results.csv",
        DATA / "evaluation/hybrid_ranking_results.csv",
        DATA / "evaluation/semantic_ranking_results.csv",
    ]
    assert all(path.exists() for path in required)

def test_synthetic_dataset_dimensions():
    resumes = pd.read_csv(DATA / "synthetic/resumes.csv")
    jobs = pd.read_csv(DATA / "synthetic/job_descriptions.csv")
    assert len(resumes) == 10
    assert len(jobs) == 10
    assert resumes["resume_id"].is_unique
    assert jobs["job_id"].is_unique

def test_reviewed_labels_are_valid():
    reviewed = pd.read_csv(DATA / "evaluation/reviewed_pairs.csv")
    assert len(reviewed) == 40
    assert reviewed[["resume_id", "job_id"]].duplicated().sum() == 0
    assert set(reviewed["relevance_label"].dropna().unique()).issubset({0, 1, 2})

def test_ranking_artifacts_have_valid_scores():
    for filename, score_column in [
        ("hybrid_ranking_results.csv", "hybrid_score"),
        ("semantic_ranking_results.csv", "semantic_score"),
    ]:
        df = pd.read_csv(DATA / "evaluation" / filename)
        assert len(df) == 40
        assert {"resume_id", "job_id", score_column}.issubset(df.columns)
        assert df[score_column].notna().all()
        assert df[score_column].between(0, 1).all()
        assert not df[["resume_id", "job_id"]].duplicated().any()

def test_skill_evidence_is_valid_json():
    df = pd.read_csv(DATA / "evaluation/skill_matching_results.csv")
    assert "skill_evidence" in df.columns
    for value in df["skill_evidence"].dropna():
        parsed = json.loads(value)
        assert isinstance(parsed, dict)

def test_coverage_is_between_zero_and_one():
    df = pd.read_csv(DATA / "evaluation/skill_matching_results.csv")
    assert df["required_coverage"].notna().all()
    assert df["required_coverage"].between(0, 1).all()
'''

test_path = TESTS / "test_artifacts.py"
test_path.write_text(test_code.strip() + "\n", encoding="utf-8")

print("[PASS] Test file created:", test_path)
print("Test functions:", test_code.count("def test_"))

[PASS] Test file created: /content/talentmatch/tests/test_artifacts.py
Test functions: 6


In [51]:
from pathlib import Path
import subprocess
import sys
import py_compile

ROOT = PROJECT_DIR

print("=== PYTEST ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(ROOT / "tests"), "-q"],
    cwd=ROOT,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-3000:])
print("Pytest exit code:", result.returncode)

print("\n=== APP SYNTAX ===")
app_path = ROOT / "app" / "streamlit_app.py"
try:
    py_compile.compile(str(app_path), doraise=True)
    print("[PASS] Streamlit app compiles")
except Exception as exc:
    print("[FAIL]", repr(exc))

print("\n=== DOCUMENTATION ===")
for relative in [
    "README.md",
    "docs/architecture.md",
    "requirements.txt",
    ".gitignore",
    "PROJECT_OVERVIEW.md",
]:
    path = ROOT / relative
    print(("[PASS] " if path.exists() else "[MISSING] ") + relative)

print("\n=== CORE ARTIFACTS ===")
for relative in [
    "data/synthetic/resumes.csv",
    "data/synthetic/job_descriptions.csv",
    "data/evaluation/reviewed_pairs.csv",
    "data/evaluation/FINAL_EVALUATION_REPORT.md",
    "data/evaluation/final_per_resume_metrics.csv",
    "data/evaluation/final_model_summary.csv",
]:
    path = ROOT / relative
    print(("[PASS] " if path.exists() else "[MISSING] ") + relative)

print("\n=== RESULT ===")
print("Pytest:", "PASS" if result.returncode == 0 else "FAIL")
print("Review the output above before GitHub preparation.")

=== PYTEST ===
......                                                                   [100%]
6 passed in 0.66s

Pytest exit code: 0

=== APP SYNTAX ===
[PASS] Streamlit app compiles

=== DOCUMENTATION ===
[PASS] README.md
[PASS] docs/architecture.md
[PASS] requirements.txt
[PASS] .gitignore
[PASS] PROJECT_OVERVIEW.md

=== CORE ARTIFACTS ===
[PASS] data/synthetic/resumes.csv
[PASS] data/synthetic/job_descriptions.csv
[PASS] data/evaluation/reviewed_pairs.csv
[PASS] data/evaluation/FINAL_EVALUATION_REPORT.md
[PASS] data/evaluation/final_per_resume_metrics.csv
[PASS] data/evaluation/final_model_summary.csv

=== RESULT ===
Pytest: PASS
Review the output above before GitHub preparation.


In [52]:
from pathlib import Path

ROOT = PROJECT_DIR

print("=== PROJECT FILE INVENTORY ===")
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and not any(
        part in {".git", "__pycache__", ".pytest_cache"} for part in path.parts
    ):
        print(path.relative_to(ROOT))

print("\n=== REPRODUCIBILITY CHECK ===")
checks = {
    "Streamlit app": ROOT / "app/streamlit_app.py",
    "Notebook in project": ROOT / "TALENTMATCH.ipynb",
    "Model/pipeline source directory": ROOT / "models",
    "Evaluation report": ROOT / "data/evaluation/FINAL_EVALUATION_REPORT.md",
    "Test suite": ROOT / "tests/test_artifacts.py",
}

for label, path in checks.items():
    if path.is_dir():
        files = list(path.glob("*.py"))
        print(f"[{'PASS' if files else 'WARN'}] {label}: {len(files)} Python files")
    else:
        print(f"[{'PASS' if path.exists() else 'WARN'}] {label}: {path.exists()}")

print("\nNote: a working app with saved CSVs does not by itself reproduce model generation.")

=== PROJECT FILE INVENTORY ===
.gitignore
PROJECT_AUDIT.json
PROJECT_OVERVIEW.md
README.md
annotation_rubric.md
app/streamlit_app.py
data/evaluation/FINAL_EVALUATION_REPORT.md
data/evaluation/final_model_summary.csv
data/evaluation/final_per_resume_metrics.csv
data/evaluation/hybrid_ranking_results.csv
data/evaluation/model_comparison_metrics.csv
data/evaluation/model_comparison_metrics_40_pairs.csv
data/evaluation/model_comparison_metrics_corrected.csv
data/evaluation/model_comparison_metrics_final.csv
data/evaluation/model_comparison_pair_scores.csv
data/evaluation/model_comparison_scores_40_pairs.csv
data/evaluation/remaining_pairs_for_review.csv
data/evaluation/review_batch_20.csv
data/evaluation/reviewed_pairs.csv
data/evaluation/semantic_ranking_results.csv
data/evaluation/semantic_similarity_scores.csv
data/evaluation/skill_matching_results.csv
data/evaluation/tfidf_metrics_by_resume.csv
data/evaluation/tfidf_pair_scores.csv
data/synthetic/job_descriptions.csv
data/synthetic/res

In [53]:
from pathlib import Path

ROOT = PROJECT_DIR

gitignore = """
# Secrets and environment files
.env
.env.*
!.env.example
*.pem
*.key
credentials.json
service-account*.json

# Python
__pycache__/
*.py[cod]
.pytest_cache/
.mypy_cache/
.ruff_cache/
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Model and experiment artifacts
.cache/
huggingface/
wandb/
mlruns/
lightning_logs/
*.pt
*.pth
*.ckpt
*.safetensors

# OS and editor
.DS_Store
Thumbs.db
.vscode/
.idea/

# Logs and temporary files
*.log
*.tmp
"""

path = ROOT / ".gitignore"
path.write_text(gitignore.strip() + "\n", encoding="utf-8")

print("[PASS] .gitignore updated")
print(path.read_text(encoding="utf-8"))

[PASS] .gitignore updated
# Secrets and environment files
.env
.env.*
!.env.example
*.pem
*.key
credentials.json
service-account*.json

# Python
__pycache__/
*.py[cod]
.pytest_cache/
.mypy_cache/
.ruff_cache/
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Model and experiment artifacts
.cache/
huggingface/
wandb/
mlruns/
lightning_logs/
*.pt
*.pth
*.ckpt
*.safetensors

# OS and editor
.DS_Store
Thumbs.db
.vscode/
.idea/

# Logs and temporary files
*.log
*.tmp



In [54]:
from pathlib import Path
import re

ROOT = PROJECT_DIR

patterns = {
    "Possible API key assignment": re.compile(
        r"""(?i)\b(api[_-]?key|secret[_-]?key|access[_-]?token)\b\s*[:=]\s*["'][^"']{12,}["']"""
    ),
    "GitHub personal access token": re.compile(r"\bgh[pousr]_[A-Za-z0-9_]{20,}\b"),
    "OpenAI-style key": re.compile(r"\bsk-[A-Za-z0-9_-]{20,}\b"),
    "AWS access key": re.compile(r"\bAKIA[0-9A-Z]{16}\b"),
}

skip_parts = {".git", "__pycache__", ".pytest_cache", ".venv", "venv"}
text_suffixes = {".py", ".md", ".txt", ".json", ".yaml", ".yml", ".toml", ".ini", ".csv", ".ipynb"}

secret_hits = []
placeholder_hits = []

for path in ROOT.rglob("*"):
    if not path.is_file() or any(part in skip_parts for part in path.parts):
        continue
    if path.name == ".gitignore" or path.suffix.lower() not in text_suffixes:
        continue

    try:
        content = path.read_text(encoding="utf-8")
    except Exception:
        continue

    for label, pattern in patterns.items():
        for match in pattern.finditer(content):
            line_number = content[:match.start()].count("\n") + 1
            secret_hits.append((str(path.relative_to(ROOT)), line_number, label))

    if "<YOUR_REPOSITORY_URL>" in content:
        placeholder_hits.append(str(path.relative_to(ROOT)))

print("=== SECRET SCAN ===")
if secret_hits:
    for filename, line, label in secret_hits:
        print(f"[REVIEW] {filename}:{line} — {label} (value hidden)")
else:
    print("[PASS] No matches from configured secret patterns")

print("\n=== README PLACEHOLDERS ===")
if placeholder_hits:
    for filename in placeholder_hits:
        print("[REVIEW] Replace repository URL placeholder in:", filename)
else:
    print("[PASS] No repository URL placeholder found")

print("\nThis is a basic pattern scan, not a guarantee that no secrets exist.")

=== SECRET SCAN ===
[PASS] No matches from configured secret patterns

=== README PLACEHOLDERS ===
[REVIEW] Replace repository URL placeholder in: README.md

This is a basic pattern scan, not a guarantee that no secrets exist.


In [55]:
from pathlib import Path
import zipfile

ROOT = PROJECT_DIR
ZIP_PATH = Path("/content/TALENTMATCH_portfolio_backup.zip")

excluded_parts = {
    ".git", "__pycache__", ".pytest_cache", ".venv", "venv",
    ".mypy_cache", ".ruff_cache"
}

included = 0

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ROOT.rglob("*"):
        if not path.is_file():
            continue
        if any(part in excluded_parts for part in path.parts):
            continue
        if path.name.endswith((".pyc", ".log")):
            continue

        archive.write(path, path.relative_to(ROOT.parent))
        included += 1

print("[PASS] Backup created:", ZIP_PATH)
print("Files included:", included)
print("Size:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2), "MB")

from google.colab import files
files.download(str(ZIP_PATH))

[PASS] Backup created: /content/TALENTMATCH_portfolio_backup.zip
Files included: 29
Size: 0.03 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
from pathlib import Path

root = Path("/content")
notebooks = [
    p for p in root.rglob("*.ipynb")
    if "site-packages" not in str(p)
    and ".ipynb_checkpoints" not in str(p)
]

print("=== NOTEBOOKS FOUND ===")
if notebooks:
    for i, path in enumerate(notebooks, 1):
        print(f"{i}. {path}")
else:
    print("No notebooks found under /content.")

=== NOTEBOOKS FOUND ===
No notebooks found under /content.


In [57]:
from pathlib import Path

root = Path("/content")
excluded = {"site-packages", "dist-packages", ".cache", ".ipython"}

candidates = []
for path in root.rglob("*.ipynb"):
    if any(part in excluded for part in path.parts):
        continue
    if ".ipynb_checkpoints" in path.parts:
        continue
    candidates.append(path)

print("=== NOTEBOOK SEARCH ===")
if candidates:
    for i, path in enumerate(sorted(candidates), 1):
        print(f"{i}. {path} ({path.stat().st_size:,} bytes)")
else:
    print("[WARN] No notebook found under /content")

print("\nExpected project destination:")
print("Project root / TALENTMATCH.ipynb")

=== NOTEBOOK SEARCH ===
[WARN] No notebook found under /content

Expected project destination:
/content/talentmatch/TALENTMATCH.ipynb


In [58]:
from pathlib import Path

project = PROJECT_DIR

print("=== PYTHON SOURCE INVENTORY ===")
python_files = [
    p for p in project.rglob("*.py")
    if "__pycache__" not in p.parts
]

if python_files:
    for path in python_files:
        print(f"\n--- {path.relative_to(project)} ---")
        print(f"Size: {path.stat().st_size:,} bytes")
        if path.name != "streamlit_app.py":
            print(path.read_text(errors="replace")[:1200])
else:
    print("[WARN] No Python source files found")

print("\n=== NOTEBOOK STATUS ===")
notebook = project / "TALENTMATCH.ipynb"
print("Project notebook exists:", notebook.exists())

=== PYTHON SOURCE INVENTORY ===

--- tests/test_artifacts.py ---
Size: 2,152 bytes
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/talentmatch")
DATA = ROOT / "data"

def test_required_project_files_exist():
    required = [
        DATA / "synthetic/resumes.csv",
        DATA / "synthetic/job_descriptions.csv",
        DATA / "evaluation/reviewed_pairs.csv",
        DATA / "evaluation/skill_matching_results.csv",
        DATA / "evaluation/hybrid_ranking_results.csv",
        DATA / "evaluation/semantic_ranking_results.csv",
    ]
    assert all(path.exists() for path in required)

def test_synthetic_dataset_dimensions():
    resumes = pd.read_csv(DATA / "synthetic/resumes.csv")
    jobs = pd.read_csv(DATA / "synthetic/job_descriptions.csv")
    assert len(resumes) == 10
    assert len(jobs) == 10
    assert resumes["resume_id"].is_unique
    assert jobs["job_id"].is_unique

def test_reviewed_labels_are_valid():
    reviewed = pd.read_csv(DATA / "evalua

In [59]:
from pathlib import Path
import pandas as pd

project = PROJECT_DIR

files = [
    "data/synthetic/resumes.csv",
    "data/synthetic/job_descriptions.csv",
    "data/evaluation/reviewed_pairs.csv",
    "data/evaluation/semantic_ranking_results.csv",
    "data/evaluation/hybrid_ranking_results.csv",
    "data/evaluation/skill_matching_results.csv",
    "data/evaluation/final_model_summary.csv",
]

print("=== DATA FILE VALIDATION ===")

for relative_path in files:
    path = project / relative_path
    if not path.exists():
        print(f"[FAIL] {relative_path} — missing")
        continue

    try:
        df = pd.read_csv(path)
        print(f"\n[PASS] {relative_path}")
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
    except Exception as e:
        print(f"[FAIL] {relative_path} — {e}")

app = project / "app/streamlit_app.py"
print("\n=== APP INPUT REFERENCES ===")
if app.exists():
    content = app.read_text(errors="replace")
    for line in content.splitlines():
        if ".csv" in line or "read_csv" in line:
            print(line.strip())
else:
    print("[FAIL] Streamlit app missing")

=== DATA FILE VALIDATION ===

[PASS] data/synthetic/resumes.csv
Shape: (10, 3)
Columns: ['resume_id', 'profile', 'resume_text']

[PASS] data/synthetic/job_descriptions.csv
Shape: (10, 3)
Columns: ['job_id', 'job_title', 'job_description']

[PASS] data/evaluation/reviewed_pairs.csv
Shape: (40, 4)
Columns: ['resume_id', 'job_id', 'relevance_label', 'label_reason']

[PASS] data/evaluation/semantic_ranking_results.csv
Shape: (40, 11)
Columns: ['resume_id', 'job_id', 'matched_required', 'missing_required', 'matched_preferred', 'required_coverage', 'semantic_score', 'hybrid_score', 'skill_evidence', 'evidence_explanation', 'relevance_label']

[PASS] data/evaluation/hybrid_ranking_results.csv
Shape: (40, 11)
Columns: ['resume_id', 'job_id', 'matched_required', 'missing_required', 'matched_preferred', 'required_coverage', 'semantic_score', 'hybrid_score', 'skill_evidence', 'evidence_explanation', 'relevance_label']

[PASS] data/evaluation/skill_matching_results.csv
Shape: (40, 10)
Columns: ['r

In [60]:
from pathlib import Path
import subprocess

project = PROJECT_DIR

def run(command):
    result = subprocess.run(
        command,
        cwd=project,
        capture_output=True,
        text=True
    )
    print("$", " ".join(command))
    print(result.stdout.strip() or result.stderr.strip() or "(no output)")
    return result.returncode

print("=== GIT READINESS ===")

git_check = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)
print(git_check.stdout.strip() or git_check.stderr.strip())

if (project / ".git").exists():
    print("\n[INFO] Git repository already initialized")
    run(["git", "status", "--short"])
else:
    print("\n[INFO] Git repository not initialized yet")

print("\n=== KEY PROJECT FILES ===")
for relative_path in [
    "README.md",
    ".gitignore",
    "PROJECT_OVERVIEW.md",
    "docs/architecture.md",
    "app/streamlit_app.py",
    "tests/test_artifacts.py",
    "TALENTMATCH.ipynb",
]:
    path = project / relative_path
    print(f"{'[PASS]' if path.exists() else '[WARN]'} {relative_path}")

=== GIT READINESS ===
git version 2.43.0

[INFO] Git repository not initialized yet

=== KEY PROJECT FILES ===
[PASS] README.md
[PASS] .gitignore
[PASS] PROJECT_OVERVIEW.md
[PASS] docs/architecture.md
[PASS] app/streamlit_app.py
[PASS] tests/test_artifacts.py
[WARN] TALENTMATCH.ipynb


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")
matches = []

for path in drive_root.rglob("*.ipynb"):
    if ".ipynb_checkpoints" in path.parts:
        continue
    if any(part.startswith(".") for part in path.relative_to(drive_root).parts):
        continue
    matches.append(path)

print("=== NOTEBOOKS FOUND IN GOOGLE DRIVE ===")

if matches:
    for i, path in enumerate(matches, 1):
        print(f"{i}. {path}")
else:
    print("[WARN] No notebooks found in MyDrive")